In [1]:
# === Cell 0: paths & config ===
MANIFEST_PATIENT = "/hpcnfs/home/ieo7627/MANIFEST_TEST_TITAN.csv"  # cambia se serve

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")

# === Cell 1: utils ===
import numpy as np
import pandas as pd
import os
import random
import ast
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from torch.utils.data import Dataset, DataLoader, Subset

def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def parse_list(x):
    if isinstance(x, list): return x
    try: return ast.literal_eval(str(x))
    except: return []

def to_numpy_f32(x):
    if isinstance(x, torch.Tensor):
        return x.detach().to(torch.float32).cpu().numpy()
    if isinstance(x, np.ndarray):
        return x.astype(np.float32, copy=False)
    if isinstance(x, (list, tuple)):
        arr = [to_numpy_f32(t) for t in x]
        arr = [t for t in arr if t is not None]
        if not arr: return None
        return np.concatenate(arr, 0) if arr[0].ndim==2 else np.stack(arr, 0)
    if isinstance(x, dict):
        for k in ("features","feats","X","x","embeddings","data"):
            if k in x: return to_numpy_f32(x[k])
        if len(x)==1: return to_numpy_f32(next(iter(x.values())))
        return None
    return None

def load_pt_matrix(pt_path, D_expected=768):
    if not os.path.exists(pt_path):
        return None
    try:
        obj = torch.load(pt_path, map_location="cpu")
    except Exception as e:
        print(f"[SKIP load] {pt_path}: {e}")
        return None
    X = to_numpy_f32(obj)
    if X is None:
        print(f"[WARN] {pt_path}: formato non riconosciuto"); return None
    if X.ndim == 1: X = X[None, :]
    if X.shape[1] != D_expected:
        print(f"[WARN] {pt_path}: D={X.shape[1]} != {D_expected}; skipping"); return None
    return X

# === Cell 2: PatientBagsDataset ===
import hashlib
class PatientBagsDataset(Dataset):
    def __init__(self, manifest_csv,
                 norm_mode="slide_center_l2",
                 sample_per_patient=2000,
                 centroid_frac=0.85,
                 kmeans_k=96,
                 random_state=42,
                 # kcenter_frac=None,           # se None usa centroid_frac
                 # diversity_mode="farthest",   # "farthest" (consigliato) | "random"
                 # diversity_temp=0.75
                ):
        self.df = pd.read_csv(manifest_csv)
        self.df["feat_paths_list"] = self.df["feat_paths"].apply(parse_list)
        self.norm_mode = norm_mode
        self.sample_per_patient = sample_per_patient
        self.centroid_frac = centroid_frac
        self.kmeans_k = kmeans_k
        self.rs = np.random.RandomState(random_state)
        self.split = "train"                 # "train" o "val" (impostalo quando crei ds)
        self.base_seed = 42                  # o quello che già usi
        self.fold = 0                        # impostalo da fuori per riproducibilità per-fold

        # cache val e rotazione train
        self.val_idx_cache = {}              # {patient_id: np.ndarray di indici fissi per VAL}
        self.train_perm = {}                 # {patient_id: permutazione intera degli indici tile}
        self.train_ptr  = {}                 # {patient_id: puntatore corrente nella permutazione}
        self._epoch = 0

        # coverage (debug)
        self._cover = {}                     # {patient_id: np.ndarray bool visti}
        # self.kcenter_frac  = centroid_frac if kcenter_frac is None else float(kcenter_frac)
        # self.diversity_mode = diversity_mode
        # self.diversity_temp = float(diversity_temp)
        
    def set_epoch(self, e: int):
        self._epoch = int(e)

    def set_split(self, split: str, fold: int = 0):
        assert split in ("train", "val")
        self.split = split
        self.fold = int(fold)
        # ---------- K-CENTER (FPS) + DIVERSITY HELPERS ----------

    @staticmethod
    def _pairwise_sq_dists(A, B):
        # A: (Na, D), B: (Nb, D) → (Na, Nb) con ||a-b||^2
        # stabile e veloce su float32
        AA = (A*A).sum(axis=1, keepdims=True)
        BB = (B*B).sum(axis=1, keepdims=True).T
        AB = A @ B.T
        D2 = np.maximum(AA + BB - 2.0*AB, 0.0)
        return D2

    def _kcenter_fps(self, X, k, rng=None, init_idx=None):
        """
        K-center greedy (farthest point sampling):
        - parte da init_idx (o dal punto a norma massima se None)
        - aggiunge ogni volta il punto più lontano dal set corrente
        X: (N, D) float32
        k: numero di punti da selezionare
        """
        N = X.shape[0]
        if k <= 0 or N == 0:
            return np.zeros((0,), dtype=int)
        k = min(k, N)

        # inizializzazione
        if init_idx is None:
            # primo punto: massimo della norma (stabile) o random deterministico
            norms = np.linalg.norm(X, axis=1)
            start = int(norms.argmax())
        else:
            start = int(init_idx)

        selected = [start]
        # mantieni le distanze minime di ogni punto al set selezionato
        d2_min = self._pairwise_sq_dists(X, X[[start], :]).reshape(-1)

        for _ in range(1, k):
            # scegli il più lontano dai selezionati
            nxt = int(np.argmax(d2_min))
            selected.append(nxt)
            # aggiorna d2_min con la nuova colonna
            d2_new = self._pairwise_sq_dists(X, X[[nxt], :]).reshape(-1)
            # distanza al set = min distanza a uno dei selezionati
            d2_min = np.minimum(d2_min, d2_new)

        return np.array(selected, dtype=int)

    def _diversity_fill(self, X, pool_idx, already_sel, m, rng, temp=0.5):
        """
        Riempi con punti "diversi" dal set già selezionato.
        Strategy 'farthest-softmax': probabilità ∝ softmax( min_dist^2 / temp )
        Se m<=0 o pool vuoto, ritorna [].
        """
        if m <= 0 or len(pool_idx) == 0:
            return np.zeros((0,), dtype=int)

        # distanza di ciascun pool-point al set selezionato
        X_pool = X[pool_idx]
        X_sel  = X[already_sel] if len(already_sel) else None
        if X_sel is None or X_sel.shape[0] == 0:
            # nessun selezionato: puro random (deterministico)
            return rng.choice(pool_idx, size=min(m, len(pool_idx)), replace=False)

        D2 = self._pairwise_sq_dists(X_pool, X_sel)   # (|pool|, |sel|)
        d2_min = D2.min(axis=1)                       # min distanza al set = diversità

        # softmax con temperatura
        s = d2_min / max(1e-6, float(temp))
        s -= s.max()  # stabilità
        p = np.exp(s)
        p /= p.sum() if p.sum() > 0 else 1.0

        m = min(m, len(pool_idx))
        # campionamento senza rimpiazzo via weighted sampling (approssimazione iterativa)
        chosen_rel = []
        avail = np.arange(len(pool_idx))
        probs = p.copy()
        for _ in range(m):
            j = rng.choice(avail, p=probs[avail]/probs[avail].sum())
            chosen_rel.append(int(j))
            # rimuovi j
            avail = avail[avail != j]
            # opzionale: aggiorna le probabilità penalizzando i vicini di j
            # qui manteniamo semplice (buono abbastanza in pratica)

        return pool_idx[np.array(chosen_rel, dtype=int)]

    def _sample_centroids_random(self, X_pool, n_cent, n_rand, rng=None):
        N = X_pool.shape[0]
        if N == 0:
            return np.zeros((0,), dtype=int)

        # # CLAMP su k per evitare cluster inutili
        # k_base = getattr(self, "kmeans_k", 32)
        # k = min(k_base, max(2, N // 10), N)
        k_min,k_max=16,96
        k=int(np.clip(int(round(np.sqrt(N))),k_min,min(k_max,N)))
        # random_state deterministico se rng è passato
        rs = None if rng is None else int(rng.randint(0, 2**31-1))

        # --- KMeans ---
        from sklearn.cluster import KMeans
        km = KMeans(n_clusters=k, n_init='auto', random_state=rs)
        km.fit(X_pool)
        labels = km.labels_
        centers = km.cluster_centers_

        # indice della tile più vicina ad ogni centro
        nearest = []
        for c in range(k):
            idx_c = np.where(labels == c)[0]
            if idx_c.size == 0: 
                continue
            dif = X_pool[idx_c] - centers[c]
            d2 = np.einsum("ij,ij->i", dif, dif)
            nearest.append(idx_c[int(d2.argmin())])
        nearest = np.array(nearest, dtype=int)

        # prendi n_cent centroidi
        if nearest.size > n_cent > 0:
            sel_cent = (rng.choice(nearest, size=n_cent, replace=False) if rng is not None
                        else np.random.choice(nearest, size=n_cent, replace=False))
        else:
            sel_cent = nearest[:n_cent] if n_cent > 0 else np.zeros((0,), dtype=int)

        # random dal resto
        mask = np.ones(N, dtype=bool)
        mask[sel_cent] = False
        rem = np.where(mask)[0]
        n_rand = max(0, n_rand)
        if rem.size > 0 and n_rand > 0:
            sel_rand = (rng.choice(rem, size=min(n_rand, rem.size), replace=False) if rng is not None
                        else np.random.choice(rem, size=min(n_rand, rem.size), replace=False))
        else:
            sel_rand = np.zeros((0,), dtype=int)

        sel = np.concatenate([sel_cent, sel_rand])
        # se per qualsiasi motivo sono meno di (n_cent+n_rand), completa dal resto
        need = (n_cent + n_rand) - sel.size
        if need > 0 and rem.size > 0:
            extra = (rng.choice(rem, size=min(need, rem.size), replace=False) if rng is not None
                    else np.random.choice(rem, size=min(need, rem.size), replace=False))
            sel = np.unique(np.concatenate([sel, extra]))

        return sel[:(n_cent + n_rand)]
    

        
    def _select_indices_with_rng(self, X_full, idx_pool, rng, M_target):
        # scegli quanti centroidi/random vuoi come fai ora
        n_cent = int(round(self.centroid_frac * M_target))
        n_rand = M_target - n_cent
        # se hai X_full (features) disponibile, lavora sul sottoinsieme
        if X_full is not None:
            X_pool = X_full[idx_pool]
            rel = self._sample_centroids_random(X_pool, n_cent, n_rand, rng=rng)  # vedi patch sotto
            return idx_pool[rel]
        # fallback: puro shuffle deterministico
        idx = idx_pool.copy()
        rng.shuffle(idx)
        return idx[:M_target]
    
    def _fixed_val_indices(self, pid, X, rng, M_target):
        if not hasattr(self, "val_idx_cache"):
            self.val_idx_cache = {}
        if pid not in self.val_idx_cache:
            idx_pool = np.arange(X.shape[0])
            chosen   = self._select_indices_with_rng(X, idx_pool, rng, M_target)  # <<< passa X, non pid
            self.val_idx_cache[pid] = chosen
        return self.val_idx_cache[pid]


    
    def _next_block_train(self, pid, N, block):
        # inizializza permutazione per paziente
        if (pid not in self.train_perm) or (self.train_perm[pid].size != N):
            rng = np.random.default_rng((hash((str(pid), self.base_seed, self.fold)) & 0xffffffff))
            self.train_perm[pid] = rng.permutation(N)
            self.train_ptr[pid]  = 0
        # finestra scorrevole (rotazione circolare)
        p  = self.train_ptr[pid]
        idx = self.train_perm[pid]
        if p + block <= N:
            pool = idx[p:p+block]
        else:
            k = (p + block) - N
            pool = np.concatenate([idx[p:], idx[:k]])
        # stride “aggressivo ma non tutto”: metà pool
        stride = max(1, block // 2)
        self.train_ptr[pid] = (p + stride) % N
        return pool

    
    def coverage_report(self):
        return {pid: float(c.mean()) for pid, c in self._cover.items()} if self._cover else {}

    def __len__(self):
        return len(self.df)

    @staticmethod
    def _l2_norm_rows(X, eps=1e-8):
        n = np.linalg.norm(X, axis=1, keepdims=True)
        return X / np.maximum(n, eps)

    def _apply_norm(self, X_list):
        if self.norm_mode == "none":
            return np.concatenate(X_list, 0)
        if self.norm_mode == "tile_l2":
            return np.concatenate([self._l2_norm_rows(X) for X in X_list], 0)
        if self.norm_mode == "slide_center":
            return np.concatenate([X - X.mean(0, keepdims=True) for X in X_list], 0)
        if self.norm_mode == "slide_center_l2":
            return np.concatenate([self._l2_norm_rows(X - X.mean(0, keepdims=True)) for X in X_list], 0)
        if self.norm_mode == "slide_zscore":
            return np.concatenate([(X - X.mean(0, keepdims=True))/(X.std(0, keepdims=True)+1e-6) for X in X_list], 0)
        raise ValueError(f"Unknown norm_mode: {self.norm_mode}")


    def __getitem__(self, i):
        
        def _stable_seed(pid, fold, base_seed):
            key = f"{pid}_{fold}_{base_seed}"
            h = hashlib.md5(key.encode("utf-8")).hexdigest()
            # prendi 8 hex (32 bit)
            return int(h[:8], 16)
        row = self.df.iloc[i]
        pid = row["patient_id"]
        # y_bin = int(row["y_bin"]) if pd.notna(row["y_bin"]) else None
        # # has_cont = bool(row["has_cont"]) if "has_cont" in row else False
        # # y_cont = float(row["y_cont"]) if ("y_cont" in row and pd.notna(row["y_cont"])) else None
        # y_cont_val = row.get("y_cont")
        # has_cont = pd.notna(y_cont_val)
        # y_cont = float(y_cont_val) if has_cont else None
        y_bin_val = row.get("y_bin", None)
        y_bin = int(y_bin_val) if (y_bin_val is not None and pd.notna(y_bin_val)) else None
        
        y_cont_val = row.get("y_cont", None)
        has_cont = (y_cont_val is not None and pd.notna(y_cont_val))
        y_cont = float(y_cont_val) if has_cont else None
        X_list, n_tiles_raw = [], 0
        for pt in row["feat_paths_list"]:
            if not isinstance(pt,str) or not os.path.exists(pt): continue
            X = load_pt_matrix(pt)
            if X is None: continue
            n_tiles_raw += X.shape[0]; X_list.append(X)
        if len(X_list)==0: X = np.zeros((0,768),np.float32)
        else: X = self._apply_norm(X_list)

        N_all = X.shape[0]
        M_target = min(self.sample_per_patient, N_all) if N_all>0 else 0

        # default per il caso M_target==0
        sel_idx = np.zeros((0,), dtype=np.int32)
        N_all_orig = N_all

        if M_target > 0:
            if getattr(self, "split", "train") == "val":
                # VAL deterministico (versione con KMeans che hai scelto)
                # rng_val = np.random.RandomState((hash((pid, self.fold, self.base_seed)) & 0xffffffff))
                seed = _stable_seed(pid, self.fold, self.base_seed)
                rng_val = np.random.RandomState(seed)
                idx = self._fixed_val_indices(pid, X, rng_val, M_target)  # PASSI X
            else:
                # TRAIN: rotazione senza sostituzione + selezione kmeans+random
                pool_size = max(M_target, int(1.5 * M_target))  # pool > target per diversità
                idx_pool = self._next_block_train(pid, N_all, pool_size)
                rng_tr = np.random.RandomState((hash((pid, getattr(self, "_epoch", 0), getattr(self, "base_seed", 42))) & 0xffffffff))
                idx = self._select_indices_with_rng(X, idx_pool, rng_tr, M_target)

            # ---- QUI: LOG COVERAGE (salva gli indici PRIMA di tagliare X) ----
            sel_idx = np.asarray(idx, dtype=np.int32)
            N_all_orig = N_all

            # taglia realmente le feature
            X = X[idx]

        bag_feats = torch.from_numpy(X)  # X è già float32

        return {
            "bag_feats": bag_feats,
            "patient_id": pid,
            "y_bin": torch.tensor([y_bin], dtype=torch.float32) if y_bin is not None else None,
            "y_cont": torch.tensor([y_cont/100.0], dtype=torch.float32) if y_cont is not None else None,
            "has_cont": torch.tensor([1.0 if has_cont else 0.0], dtype=torch.float32),
            "n_tiles_raw": n_tiles_raw,
            "n_wsi": int(row.get("n_wsi", 0)),
            # --- campi per coverage nel MAIN ---
            "sel_idx": torch.from_numpy(sel_idx),                 # (M_target,) int32
            "n_all": torch.tensor(N_all_orig, dtype=torch.int32), # scalar int32
        }
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# --- utils robusti ---
def coerce_binary(col):
    s = col.copy()
    s = s.replace({True: 1, False: 0})
    s = s.astype(str).str.strip().str.upper().replace({
        "HN": 1, "HIGH": 1, "YES": 1, "Y": 1, "TRUE": 1, "1": 1,
        "LN": 0, "LOW": 0,  "NO": 0,  "N": 0, "FALSE": 0, "0": 0,
        "NA": np.nan, "N/A": np.nan, "NONE": np.nan, "": np.nan, "NAN": np.nan
    })
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return s

dfp = pd.read_csv(MANIFEST_PATIENT)
dfp["feat_paths_list"] = dfp["feat_paths"].apply(parse_list)
dfp["y_bin_num"]  = coerce_binary(dfp.get("y_bin"))

# 2) tieni SOLO chi ha y_bin per training/val
mask_keep = dfp["y_bin_num"].notna()
dropped = dfp.loc[~mask_keep, ["patient_id","y_bin"]]
if len(dropped):
    print(f"Esclusi {len(dropped)} pazienti senza y_bin.")

df_lab = dfp.loc[mask_keep].reset_index(drop=True)
df_lab["y_bin_num"] = df_lab["y_bin_num"].astype(int)

# 3) folds stratificati su y_bin
y   = df_lab["y_bin_num"].values
grp = df_lab["patient_id"].astype(str).values
has_cont = dfp["has_cont"].astype(bool).values
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=58)
folds = list(sgkf.split(np.zeros_like(y), y, grp))

# 4) ricrea il dataset SOLO con df_lab
MANIFEST_LABELED = MANIFEST_PATIENT.replace(".csv","_labeled.csv")
df_lab.to_csv(MANIFEST_LABELED, index=False)

# ds = PatientBagsDataset(
#     manifest_csv=MANIFEST_LABELED,
#     norm_mode="slide_zscore",
#     sample_per_patient=4000,   # per Jupyter test
#     # ... altri tuoi parametri
# )
def collate_patient(batch):
    if len(batch)==1: return batch[0]
#     return {k:[b[k] for b in batch] for k in batch[0]}
# loader = DataLoader(ds,batch_size=1,shuffle=True,num_workers=2,collate_fn=collate_patient)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ========== Utils ==========
def xavier_(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None: nn.init.zeros_(m.bias)

class GatedAttn(nn.Module):
    """Gated Attention (Ilse et al.), backbone di CLAM."""
    def __init__(self, in_dim=384, attn_dim=128, dropout=0.0):
        super().__init__()
        self.V = nn.Linear(in_dim, attn_dim, bias=True)
        self.U = nn.Linear(in_dim, attn_dim, bias=True)
        self.w = nn.Linear(attn_dim, 1, bias=False)
        self.tanh = nn.Tanh()
        self.sigmoid = nn.Sigmoid()
        self.drop = nn.Dropout(dropout)
        self.apply(xavier_)

    def forward(self, H):           # H: (N, D)
        Vh = self.tanh(self.V(H))   # (N, A)
        Uh = self.sigmoid(self.U(H))
        A = self.drop(Vh * Uh)      # (N, A)
        A = self.w(A).squeeze(-1)   # (N,)
        A = torch.softmax(A, dim=0) # somma=1 nel bag
        Z = torch.sum(H * A.unsqueeze(-1), dim=0)  # (D,)
        return Z, A

# ---------- 1) CLAM ----------
class CLAMHead(nn.Module):
    """
    CLAM single-branch (binario). Attenzione gated + istanza-classifier (per instance-loss opzionale).
    Restituisce: bag_emb, attn, logit_bin, y_cont, inst_logits.
    """
    def __init__(self, in_dim=384, attn_dim=128, dropout=0.1):
        super().__init__()
        self.attn = GatedAttn(in_dim=in_dim, attn_dim=attn_dim, dropout=dropout)
        self.fc_bin = nn.Linear(in_dim, 1)  # bag -> logit binario
        self.fc_reg = nn.Linear(in_dim, 1)  # bag -> valore continuo
        self.inst_cls = nn.Linear(in_dim, 1) # per instance-loss (top-k)
        self.apply(xavier_)

    def forward(self, H):  # H: (N, D)
        bag_emb, attn = self.attn(H)                    # (D,), (N,)
        logit_bin = self.fc_bin(bag_emb).squeeze(-1)    # scalar
        y_cont   = self.fc_reg(bag_emb).squeeze(-1)     # scalar
        inst_logits = self.inst_cls(H).squeeze(-1)      # (N,)
        return {
            "bag_emb": bag_emb,
            "attn": attn,
            "logit_bin": logit_bin,
            "y_cont": y_cont,
            "inst_logits": inst_logits
        }

@torch.no_grad()
def clam_topk_indices(attn, k=8):
    k = min(k, attn.numel())
    return torch.topk(attn, k=k, largest=True).indices

def clam_instance_loss(inst_logits, attn, y_bin, k=8):
    """
    Instance-loss semplice stile CLAM:
    - se y=1 -> top-k (per attn) verso positivo
    - se y=0 -> top-k verso negativo
    """
    idx = clam_topk_indices(attn, k)
    logits = inst_logits[idx]                    # (k,)
    target = torch.full_like(logits, float(y_bin))
    return F.binary_cross_entropy_with_logits(logits, target)


# ---------- 2) DSMIL ----------
def _xavier_(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
class DSMILHead(nn.Module):
    """
    DSMIL (dual-stream):
    - stream istanza: inst_logits per tile
    - stream bag: attenzione guidata dalla top-istanza (query = h_top)
    - fusione: convex combo tra p_inst(max) e p_bag, poi riconvertita a logit
    """
    def __init__(self, in_dim=384, attn_dim=128, alpha=0.5, dropout=0.1, use_v=True, fuse_mode="prob"):
        super().__init__()
        self.inst_cls = nn.Linear(in_dim, 1)        # istanza -> logit
        self.q = nn.Linear(in_dim, attn_dim)        # Q(h_top)
        self.k = nn.Linear(in_dim, attn_dim)        # K(H)
        self.v = nn.Linear(in_dim, in_dim) if use_v else nn.Identity()  # V(H)
        self.fc_bag = nn.Linear(in_dim, 1)          # bag_emb -> logit bin
        self.fc_reg = nn.Linear(in_dim, 1)          # bag_emb -> y_cont
        self.drop = nn.Dropout(dropout)
        self.alpha = float(alpha)
        assert 0.0 <= self.alpha <= 1.0
        assert fuse_mode in {"prob", "logit"}
        self.fuse_mode = fuse_mode
        self.apply(_xavier_)

    @staticmethod
    def _logit_from_prob(p, eps=1e-6):
        p = p.clamp(eps, 1 - eps)
        return torch.log(p) - torch.log1p(-p)

    def forward(self, H):            # H: (N, D)
        # ---- 1) stream istanza
        inst_logits = self.inst_cls(H).squeeze(-1)      # (N,)
        # inutile passare da sigmoid per l'argmax (monotona):
        top_index = torch.argmax(inst_logits)           # scalar

        # ---- 2) attenzione guidata da h_top
        h_top = H[top_index]                             # (D,)
        q = self.q(self.drop(h_top))                     # (A,)
        K = self.k(self.drop(H))                         # (N, A)
        scores = torch.matmul(K, q) / math.sqrt(K.size(-1))  # (N,)
        # calcola attn in float32 per stabilità con AMP/bfloat16
        attn = torch.softmax(scores.float(), dim=0).to(H.dtype)   # (N,)
        bag_emb = torch.sum(self.v(H) * attn.unsqueeze(-1), dim=0)  # (D,)

        # ---- 3) teste finali
        bag_logit = self.fc_bag(bag_emb).squeeze(-1)     # scalar
        y_cont    = self.fc_reg(bag_emb).squeeze(-1)     # scalar

        # ---- 4) fusione
        if self.fuse_mode == "prob":
            p_inst = torch.sigmoid(inst_logits).max()    # scalar
            p_bag  = torch.sigmoid(bag_logit)           # scalar
            p_mix  = self.alpha * p_inst + (1.0 - self.alpha) * p_bag
            logit_bin = self._logit_from_prob(p_mix)     # per BCEWithLogits
        else:  # "logit" (come il tuo codice attuale)
            logit_bin = self.alpha * inst_logits.max() + (1.0 - self.alpha) * bag_logit

        return {
            "bag_emb": bag_emb,
            "attn": attn,
            "logit_bin": logit_bin,
            "y_cont": y_cont,
            "inst_logits": inst_logits,
            "top_index": top_index
        }

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------
# Utils inizializzazione
# ----------------------------
def _xavier_(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

# ----------------------------
# Bias posizionale 2D relativo
# ----------------------------
class RelPosBias2D(nn.Module):
    """
    Bias posizionale 2D relativo:
    - Input: coords_norm (B, N, 2) con valori in [0,1] per ogni token (NO CLS)
    - Costruisce bias per tutte le coppie (i,j): b_ij = MLP([dx, dy, |dx|, |dy|, r, r^2])
    - Restituisce (B, n_heads, L, L) dove L = N+1 (includo CLS con bias = 0 verso/da altri)
    """
    def __init__(self, n_heads: int, hidden: int = 64):
        super().__init__()
        self.n_heads = n_heads
        self.mlp = nn.Sequential(
            nn.Linear(6, hidden), nn.GELU(),
            nn.Linear(hidden, n_heads)
        )
        self.apply(_xavier_)

    def forward(self, coords_norm: torch.Tensor) -> torch.Tensor:
        """
        coords_norm: (B, N, 2) in [0,1]
        returns: attn_bias (B, n_heads, N+1, N+1)
        """
        B, N, _ = coords_norm.shape
        if N == 0:
            # no tokens -> solo CLS (1)
            return coords_norm.new_zeros(B, self.n_heads, 1, 1)

        xy = coords_norm  # (B,N,2)
        xi = xy.unsqueeze(2)              # (B,N,1,2)
        xj = xy.unsqueeze(1)              # (B,1,N,2)
        d  = xi - xj                      # (B,N,N,2)
        dx, dy = d[..., 0], d[..., 1]     # (B,N,N)

        r2 = dx*dx + dy*dy
        r = torch.sqrt(r2 + 1e-12)
        feat = torch.stack([dx, dy, dx.abs(), dy.abs(), r, r2], dim=-1)  # (B,N,N,6)

        bh = self.mlp(feat)  # (B, N, N, n_heads)
        bh = bh.permute(0, 3, 1, 2).contiguous()  # (B, n_heads, N, N)

        # inserisco una riga/colonna per CLS con bias=0
        L = N + 1
        out = torch.zeros((B, self.n_heads, L, L), dtype=bh.dtype, device=bh.device)
        out[:, :, 1:, 1:] = bh  # CLS(0) senza bias
        return out

# ----------------------------
# Multi-head attention con ritorno delle mappe
# ----------------------------
class MultiheadSelfAttention(nn.Module):
    """
    MHA custom (Q=K=V=X proiettato) che:
    - supporta attn_bias (B, h, L, L) additivo
    - ritorna attn_weights per ispezione
    """
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=True)
        self.drop = nn.Dropout(dropout)
        self.apply(_xavier_)

    def forward(self, x, attn_bias=None):
        """
        x: (B, L, D)
        attn_bias: (B, h, L, L) or None
        returns: (out, attn_weights_meanHeads)
            out: (B, L, D)
            attn_weights_meanHeads: (B, L, L) (media sulle teste) in fp32
        """
        B, L, D = x.shape
        qkv = self.qkv(x)  # (B, L, 3D)
        q, k, v = qkv.chunk(3, dim=-1)  # ciascuno (B, L, D)

        # reshape per heads
        def split_heads(t):
            return t.view(B, L, self.n_heads, self.d_head).transpose(1, 2)  # (B, h, L, d)
        q = split_heads(q)
        k = split_heads(k)
        v = split_heads(v)

        # scaled dot-product
        scale = 1.0 / math.sqrt(self.d_head)
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) * scale  # (B,h,L,L)
        if attn_bias is not None:
            attn_scores = attn_scores + attn_bias

        # softmax in fp32 per stabilità
        attn_weights = F.softmax(attn_scores.float(), dim=-1)  # (B,h,L,L, fp32)
        attn_weights = attn_weights.to(v.dtype)
        attn = self.drop(attn_weights)

        out = torch.matmul(attn, v)  # (B,h,L,d)
        out = out.transpose(1, 2).contiguous().view(B, L, D)  # (B,L,D)
        out = self.proj(out)
        # media teste per ispezione (ritorno fp32)
        attn_mean = attn_weights.mean(dim=1)  # (B,L,L)
        return out, attn_mean

# ----------------------------
# Encoder layer (pre-norm) + MLP
# ----------------------------
class TransformerEncoderLayerRetAttn(nn.Module):
    def __init__(self, d_model=384, n_heads=8, dim_ff=1024, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = MultiheadSelfAttention(d_model, n_heads, dropout=dropout)
        self.drop1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_ff), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_ff, d_model)
        )
        self.drop2 = nn.Dropout(dropout)
        self.apply(_xavier_)

    def forward(self, x, attn_bias=None):
        """
        x: (B,L,D)
        attn_bias: (B,h,L,L) or None
        returns: x_out, attn_meanHeads (B,L,L)
        """
        # pre-norm
        y, attn_map = self.attn(self.norm1(x), attn_bias=attn_bias)
        x = x + self.drop1(y)
        y = self.ff(self.norm2(x))
        x = x + self.drop2(y)
        return x, attn_map

# ----------------------------
# TransMIL (completo)
# ----------------------------
class TransMILHead(nn.Module):
    """
    TransMIL completo:
    - Proiezione input -> d_model
    - Token CLS
    - K encoder layers con MHA che ritorna attn maps
    - Bias posizionale 2D relativo opzionale (slide-level)
    - Logit binario/regressione dal CLS
    - Attenzione di output = media teste della mappa CLS->token dell'ULTIMO layer
    - Top-K opzionale a monte per ridurre O(N^2)
    """
    def __init__(
        self,
        in_dim=384,
        d_model=384,
        n_heads=8,
        n_layers=2,
        dim_ff=1024,
        dropout=0.1,
        use_relpos2d=True,     # True per SlideBag; False per PatientBag
        topk_tokens: int = None  # es: 2048 per ridurre costo; None = usa tutti
    ):
        super().__init__()
        self.in_proj = nn.Identity() if in_dim == d_model else nn.Linear(in_dim, d_model)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls, std=0.02)

        self.layers = nn.ModuleList([
            TransformerEncoderLayerRetAttn(d_model, n_heads, dim_ff, dropout)
            for _ in range(n_layers)
        ])

        self.use_relpos2d = bool(use_relpos2d)
        self.relpos = RelPosBias2D(n_heads) if self.use_relpos2d else None
        self.n_heads = n_heads
        self.topk_tokens = topk_tokens

        # heads sul CLS
        self.fc_bin = nn.Linear(d_model, 1)
        # self.logit_norm = nn.LayerNorm(1)   # oppure nn.LayerNorm(1) se preferisci
        self.fc_reg = nn.Linear(d_model, 1)

        self.drop = nn.Dropout(dropout)
        self.apply(_xavier_)

    def _maybe_topk(self, H, coords_norm=None):
        """
        Riduce il numero di token (N) a top-k in base alla norma del feature vector.
        Metodo semplice/stabile per gestire bag enormi.
        Ritorna H_reduced, coords_reduced, idx_kept
        """
        if (self.topk_tokens is None) or (H.size(0) <= self.topk_tokens):
            N = H.size(0)
            idx = torch.arange(N, device=H.device)
            return H, coords_norm, idx

        # score semplice: ||h||2
        scores = torch.norm(H, dim=1)  # (N,)
        k = min(self.topk_tokens, H.size(0))
        topk = torch.topk(scores, k=k, largest=True, sorted=False).indices
        Hk = H[topk]
        ck = coords_norm[topk] if (coords_norm is not None) else None
        return Hk, ck, topk

    def forward(self, H, coords_norm=None):
        """
        H: (N, D)
        coords_norm: (N, 2) in [0,1] per slide (solo se use_relpos2d=True)
        returns dict: bag_emb, attn (N,), logit_bin, y_cont
        """
        device = H.device
        dtype  = H.dtype

        # N==0 fallback
        if H.ndim != 2 or H.size(0) == 0:
            H = torch.zeros((1, H.size(-1) if H.ndim==2 else 384), device=device, dtype=dtype)

        # top-k opzionale per O(N^2)
        Hk, ck, kept = self._maybe_topk(H, coords_norm)

        # proiezione + CLS
        X = self.in_proj(Hk)          # (Nk, d_model)
        X = self.drop(X)
        B = 1
        CLS = self.cls.expand(B, 1, X.size(-1))         # (1,1,D)
        S = torch.cat([CLS, X.unsqueeze(0)], dim=1)     # (1, 1+Nk, D)

        # bias posizionale
        if self.use_relpos2d and (ck is not None):
            attn_bias = self.relpos(ck.unsqueeze(0))    # (1, h, L, L) con L=1+Nk
        else:
            attn_bias = None

        # encoder
        attn_last = None
        Z = S
        for layer in self.layers:
            Z, attn_map = layer(Z, attn_bias=attn_bias)  # attn_map: (B, L, L) (media heads)
            attn_last = attn_map                         # tieni l'ultimo

        cls_out = Z[:, 0, :]        # (1, D)
        tok_out = Z[:, 1:, :]       # (1, Nk, D)

        # teste finali
        logit_bin = self.fc_bin(cls_out).squeeze()
        y_cont    = self.fc_reg(cls_out).squeeze()
                          
        # ----- teste finali -----
        # cls_out: (1, D)
        # logit_raw = self.fc_bin(cls_out)          # (1, 1)
        # logit_cal = self.logit_norm(logit_raw)      # (1, 1) rescalato
        # logit_bin = logit_cal.squeeze()           # scalar per BCEWithLogits

        # # regressione continua (raw, in R)
        # y_cont = self.fc_reg(cls_out).squeeze()
        

        # attenzione da restituire = CLS->token dall'ultimo layer (media heads)
        # attn_last: (1, L, L) in fp32; prendo riga CLS (0) verso i token (1:)
        attn_cls_to_tok = attn_last[:, 0, 1:].squeeze(0).float()  # (Nk,)

        # se abbiamo fatto top-k, rimappa su N con zeri dove token scartati
        if (self.topk_tokens is not None) and (kept is not None) and (kept.numel() != H.size(0)):
            attn_full = torch.zeros(H.size(0), dtype=attn_cls_to_tok.dtype, device=attn_cls_to_tok.device)
            attn_full[kept] = attn_cls_to_tok
            attn_out = attn_full
        else:
            attn_out = attn_cls_to_tok  # (N,)

        return {
            "bag_emb": cls_out.squeeze(0),   # (D,)
            "attn": attn_out,                # (N,)
            "logit_bin": logit_bin,          # scalar (logit per BCEWithLogits)
            "y_cont": y_cont                 # scalar
        }


# ---------- Factory ----------
def build_mil_head(kind: str, **kwargs) -> nn.Module:
    kind = (kind or "").lower()
    if kind == "clam":
        return CLAMHead(**kwargs)
    if kind == "dsmil":
        return DSMILHead(**kwargs)
    if kind == "transmil":
        return TransMILHead(**kwargs)
    raise ValueError(f"mil_head '{kind}' non supportato. Usa: 'clam' | 'dsmil' | 'transmil'")
import numpy as np
import random
import torch
import torch.nn.functional as F

# ----- Metriche binarie safe (monoclasse) -----
from sklearn.metrics import roc_auc_score, average_precision_score, mean_absolute_error, brier_score_loss
from scipy.stats import spearmanr, pearsonr

def bin_metrics(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)
    m = np.isfinite(y_true) & np.isfinite(y_prob)
    y, p = y_true[m], y_prob[m]
    if y.size < 2 or np.unique(y).size < 2:
        return {"AUC": np.nan, "PR-AUC": np.nan}
    return {"AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p), "Brier": brier_score_loss(y, p)}

def cont_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    y, p = y_true[m], y_pred[m]
    out = {"MAE": np.nan, "Spearman": np.nan, "Pearson": np.nan}
    if y.size >= 2:
        out["MAE"] = mean_absolute_error(y, p)
        try:
            out["Spearman"] = spearmanr(y, p).correlation
        except Exception:
            pass
        try:
            out["Pearson"] = pearsonr(y, p)[0]
        except Exception:
            pass
    return out

# ----- Bilanciamento: pos_weight -----
def compute_pos_weight(dataset, indices):
    ys = [float(dataset[i]["y_bin"]) for i in indices]
    pos = sum(ys)
    neg = len(ys) - pos
    pos_weight = (neg + 1e-6) / (pos + 1e-6)
    return torch.tensor([pos_weight], dtype=torch.float32)

# ----- Instance Dropout (drop di tile) -----
def instance_dropout(H, p=0.1, min_keep=128):
    if H.size(0) <= min_keep or p <= 0:
        return H
    keep = torch.rand(H.size(0), device=H.device) > p
    if keep.sum().item() < min_keep:
        idx = torch.randperm(H.size(0), device=H.device)[:min_keep]
        keep = torch.zeros(H.size(0), dtype=torch.bool, device=H.device); keep[idx] = True
    return H[keep]

# ----- Feature jitter (rumore gaussiano leggero) -----
def feature_jitter(H, sigma=0.02):
    if sigma <= 0: return H
    return H + sigma * torch.randn_like(H)

# ----- Entropia attenzione (regolarizzatore opzionale) -----
def attention_entropy(attn):
    a = attn.clamp_min(1e-8)
    H = -(a * a.log()).sum()
    return H / math.log(attn.numel() + 1e-8)  # ∈ [0,1]

# ----- Label smoothing binario -----
def smooth_targets(yb, eps=0.05):
    return yb*(1-eps) + 0.5*eps

# ----- Seeding serio per DataLoader -----
def make_loader_seed(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)
    def _wif(_):
        random.seed(seed); np.random.seed(seed)
    return g, _wif

from torch.nn.utils import clip_grad_norm_
def _scalar(x, default=float("nan")):
    import torch
    if x is None:
        return default
    if torch.is_tensor(x):
        return float(x.view(-1)[0].item()) if x.numel() else default
    try:
        return float(x)
    except Exception:
        return default




def _norm_pid(x):
    import pandas as pd
    try:
        if isinstance(x,(list,tuple)): x=x[0]
        if hasattr(x,'item'): x=x.item()
        if isinstance(x,(int,float)) and not(pd.isna(x)): return str(int(x))
        return str(x)
    except Exception:
        return str(x)

def run_one_epoch_v2(model, loader, phase, optimizer=None,
                     alpha_reg=0.5, pos_weight=None,
                     lambda_attn=0.0, label_smooth_eps=0.0,
                     jitter_sigma=0.02, inst_drop_p=0.1, min_keep=128,
                     use_clam_inst_loss=False, clam_topk=8,
                     device=None, max_grad_norm=2.0,lambda_inst=1e-3,coverage_acc=None):
    assert phase in {"train","val","test"}
    training = (phase == "train")
    model.train(training)

    all_bin, all_prob = [], []
    all_cont_true, all_cont_pred = [], []

    tot_loss, n_steps = 0.0, 0
    for batch in loader:
        # batch fields attesi: H (N,384), y_bin (scalar 0/1), y_cont (scalar float), has_cont (0/1)
        batch["H"] = batch["bag_feats"]
        H = batch["H"].to(device)                 # (N, D)
        y_bin_val  = _scalar(batch.get("y_bin"))
        assert np.isfinite(y_bin_val), "y_bin mancante/NaN: quel paziente non dovrebbe essere nel fold di train/val."
                # --- coverage logging nel MAIN via dict mutabile ---
        if (coverage_acc is not None) and (phase == "train"):
            pid = batch["patient_id"]
            # se per caso il collate restituisse una lista
            pid = str(pid)  # <<< coerente su tutte le epoche
            sel = batch.get("sel_idx", None)
            n_all = batch.get("n_all", None)
            if (sel is not None) and (n_all is not None):
                idx = sel.cpu().numpy()
                N   = int(n_all.item())
                buf = coverage_acc.get(pid)
                if (buf is None) or (buf.size != N):
                    buf = np.zeros(N, dtype=bool)
                    coverage_acc[pid] = buf
                if idx.size > 0:
                    buf[idx] = True

        y_cont_val = _scalar(batch.get("y_cont"), default=float("nan"))
        hasc_val   = _scalar(batch.get("has_cont"), default=float("nan"))
        # se has_cont non è valido, deducilo da y_cont (NaN => 0)
        if not np.isfinite(hasc_val):
            hasc_val = 1.0 if np.isfinite(y_cont_val) else 0.0
        
        yb       = torch.tensor([y_bin_val],  dtype=torch.float32, device=device)
        yhat     = torch.tensor([y_cont_val], dtype=torch.float32, device=device)
        has_cont = torch.tensor([hasc_val],   dtype=torch.float32, device=device)

        # Bag vuote: fallback 1xD zeros
        if H.ndim != 2 or H.size(0) == 0:
            H = torch.zeros((1, 384), dtype=torch.float32, device=device)

        # Augment/Reg only in training
        if training:
            H = instance_dropout(H, p=inst_drop_p, min_keep=min_keep)
            H = feature_jitter(H, sigma=jitter_sigma)

        out = model(H)
        logit = out["logit_bin"]
        y_cont_pred = out["y_cont"]
        attn = out["attn"]

        # --- Loss binaria con pos_weight e (opz.) label smoothing
        if label_smooth_eps > 0:
            yb_eff = smooth_targets(yb, eps=label_smooth_eps)
        else:
            yb_eff = yb
        pw = None
        if pos_weight is not None:
            pw = pos_weight.to(device)
        loss_bin = F.binary_cross_entropy_with_logits(logit.unsqueeze(0), yb_eff, pos_weight=pw)

        # --- Loss continua (mask su has_cont)
        if torch.isfinite(yhat).all() and has_cont.sum() > 0:
            loss_reg = F.smooth_l1_loss(y_cont_pred.unsqueeze(0), yhat, reduction="none")
            loss_reg = (loss_reg * has_cont).sum() / (has_cont.sum() + 1e-6)
        else:
            loss_reg = torch.tensor(0.0, device=device)

        loss_inst = torch.tensor(0.0, device=device)
        if use_clam_inst_loss and ("inst_logits" in out):
            # assicurati che clam_instance_loss sia MEDIA su tile, non somma
            loss_inst = lambda_inst * clam_instance_loss(out["inst_logits"], attn, yb.item(), k=clam_topk)

        loss_attn_reg = torch.tensor(0.0, device=device)
        if lambda_attn > 0:
            loss_attn_reg = - lambda_attn * attention_entropy(attn)

        loss = loss_bin  + loss_inst + loss_attn_reg+ alpha_reg * loss_reg

        if training and optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            if max_grad_norm is not None and max_grad_norm > 0:
                clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

        # metriche
        prob = torch.sigmoid(logit).detach().item()
        all_bin.append(yb.item()); all_prob.append(prob)
        if has_cont.item() > 0.5 and torch.isfinite(yhat).item():
            all_cont_true.append(yhat.item()); all_cont_pred.append(y_cont_pred.detach().item())

        tot_loss += loss.detach().item()
        n_steps += 1

    # aggrega metriche
    bm = bin_metrics(all_bin, all_prob)
    cm = cont_metrics(all_cont_true, all_cont_pred)
    avg_loss = tot_loss / max(n_steps, 1)
    return {"loss": avg_loss, **bm, **cm}


Esclusi 1 pazienti senza y_bin.


In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, message=r".*weights_only=False.*")

import numpy as np, torch, matplotlib.pyplot as plt, pandas as pd
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.isotonic import IsotonicRegression

# ===================== CONFIG =====================
MIL_KIND = "transmil"          # "clam" | "dsmil" | "transmil"
#SELECT_FOLD = 1            # fold rapido per test
EPOCHS = 80#20                 # poche epoche per smoke test
DROPOUT = 0.2#0.1
LR = 3e-4
WD = 1e-4
ALPHA_REG = 3.0#3.0
LABEL_SMOOTH = 0.05
JITTER_SIGMA = 0.05
INST_DROP_P = 0.1
MIN_KEEP = 512
LAMBDA_ATTN = 1e-4       # reg. entropia attenzione (molto piccolo)
LAMBDA_INST = 1e-5       # reg. istanza (molto piccolo)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === NEW: multi split & init seeds ===
SPLIT_SEEDS = [58,1,42]      # puoi aggiungere altri split-seed
INIT_SEEDS  = [42]  # semi di inizializzazione (stessi split)

SELECT_FOLD = None          # None = tutti i fold
ROLLBACK_ON_PLATEAU = True
ROLLBACK_PATIENCE   = 5
RESTART_LR_FACTOR   = 0.5
MAX_RESTARTS        = 3
IMPROVE_EPS         = 1e-4  # margine minimo per AP-lift
GUARD_GAP           = 0.25
GUARD_EPOCHS        = 10
PATIENCE            = 10

In [4]:
import pandas as pd

# 1. La tua lista di target
pazienti_target = [
    'TCGA-4T-AA8H', 'TCGA-5M-AATE', 'TCGA-AA-3956', 'TCGA-AA-A010', 
    'TCGA-AA-A01P', 'TCGA-AA-A01R', 'TCGA-AA-A01S', 'TCGA-AA-A01V', 
    'TCGA-AA-A01X', 'TCGA-AA-A024', 'TCGA-AA-A029', 'TCGA-CA-6718', 
    'TCGA-CM-5864', 'TCGA-CM-6162', 'TCGA-CM-6163', 'TCGA-CM-6168', 
    'TCGA-DM-A1D9', 'TCGA-DM-A1DB', 'TCGA-DM-A280', 'TCGA-DM-A285', 
    'TCGA-DM-A28A', 'TCGA-DM-A28E', 'TCGA-F4-6703'
]

# 2. Carica i file che compongono il dataset MERGED
file_prob = 'TCGA_HN_LN_Prob_MERGED.csv'
file_17 = 'transmil_External_TCGA_17.csv'

df_prob = pd.read_csv(file_prob)
df_17 = pd.read_csv(file_17)

# 3. Crea il set unificato dei pazienti esistenti
ids_esistenti = set(df_prob['patient_id']).union(set(df_17['patient_id']))

# 4. Trova chi manca
mancanti = [p for p in pazienti_target if p not in ids_esistenti]

print(f"Pazienti totali cercati: {len(pazienti_target)}")
print(f"Pazienti MANCANTI nel file merged: {len(mancanti)}")
print("Lista dei mancanti:")
print(mancanti)

Pazienti totali cercati: 23
Pazienti MANCANTI nel file merged: 6
Lista dei mancanti:
['TCGA-5M-AATE', 'TCGA-CM-5864', 'TCGA-CM-6162', 'TCGA-CM-6163', 'TCGA-CM-6168', 'TCGA-F4-6703']


In [5]:
import pandas as pd
import os

# Lista dei 6
pazienti_target = [
    'TCGA-5M-AATE', 'TCGA-CM-5864', 'TCGA-CM-6162', 
    'TCGA-CM-6163', 'TCGA-CM-6168', 'TCGA-F4-6703'
]

# File GDC
file_gdc = 'gdc_sample_sheet.2026-01-21.tsv'
# Directory base delle immagini sul cluster
DIRECTORY_BASE = "/hpcscratch/ieo/ieo7627/TCGA" 

df_gdc = pd.read_csv(file_gdc, sep='\t')
df_filtered = df_gdc[df_gdc['Case ID'].isin(pazienti_target)]

output_list = 'svs_list_missing_6.txt'

with open(output_list, 'w') as f:
    for _, row in df_filtered.iterrows():
        # Costruisci percorso: BASE / UUID / Filename
        path = f"{DIRECTORY_BASE}/{row['File ID']}/{row['File Name']}"
        f.write(path + '\n')

print(f"Generata lista per i 6 mancanti: {output_list}")

Generata lista per i 6 mancanti: svs_list_missing_6.txt


In [3]:
import pandas as pd
import os
import glob

# --- CONFIGURAZIONE PERCORSI ---

# 1. Dove sono finiti i file parquet dopo il tiling?
# (Percorso preso dai log che hai inviato prima)
TILE_INDEX_DIR = "/hpcscratch/ieo/ieo7627/parquet_TITAN_TCGA/more_23/tile_index/"

# 2. Dove verranno salvate le features (file .pt) in futuro?
# (Questo serve per popolare la colonna 'feat_paths')
FEATURES_OUTPUT_DIR = "/hpcscratch/ieo/ieo7627/features/more_23/"

# -------------------------------

print(f"Scansione directory: {TILE_INDEX_DIR} ...")

# Trova tutti i file .parquet nella directory
parquet_files = glob.glob(os.path.join(TILE_INDEX_DIR, "*.parquet"))

if not parquet_files:
    print("Errore: Nessun file .parquet trovato nella directory specificata!")
    exit()

manifest_rows = []

for file_path in parquet_files:
    # Esempio file_path: .../tile_index/TCGA-AA-A024-01Z-00-DX1...parquet
    filename = os.path.basename(file_path)
    stem = filename.replace('.parquet', '')  # Rimuove l'estensione
    
    # --- 1. Determina il Patient ID ---
    # Lo standard TCGA è che i primi 12 caratteri sono l'ID paziente (es. TCGA-AA-A024)
    # Se i tuoi file iniziano con dei numeri strani (es. 48293191.parquet), 
    # questo metodo fallirà e servirà una mappa di correzione.
    patient_id = stem[:12] 
    
    # --- 2. Conta le tile reali ---
    try:
        # Legge il parquet per contare le righe (ogni riga è una tile)
        df_tile = pd.read_parquet(file_path)
        n_tiles = len(df_tile)
    except Exception as e:
        print(f"Errore lettura {filename}: {e}")
        n_tiles = 0 # O salta il file con 'continue'
    
    # --- 3. Costruisci le entry del Manifest ---
    # Formattazione stringa lista come richiesto: "['valore']"
    slide_ids_str = f"['{stem}']"
    feat_paths_str = f"['{FEATURES_OUTPUT_DIR}{stem}.pt']"
    
    row = {
        'patient_id': patient_id,
        'n_wsi': 1,
        'slide_ids': slide_ids_str,
        'feat_paths': feat_paths_str,
        'n_tiles_total': n_tiles,
        'feature_dim_mode': "768",
        'all_feat_exist': True
    }
    
    manifest_rows.append(row)

# Crea DataFrame
df_manifest = pd.DataFrame(manifest_rows)

# Ordina le colonne come nel file originale
cols_order = ['patient_id', 'n_wsi', 'slide_ids', 'feat_paths', 
              'n_tiles_total', 'feature_dim_mode', 'all_feat_exist']
df_manifest = df_manifest[cols_order]

# Salva
output_csv = "TCGA_MANIFEST_FROM_DIR.csv"
df_manifest.to_csv(output_csv, index=False)

print("-" * 30)
print(f"Manifest creato con successo: {output_csv}")
print(f"Totale slide processate: {len(df_manifest)}")
print(df_manifest.head())

Scansione directory: /hpcscratch/ieo/ieo7627/parquet_TITAN_TCGA/more_23/tile_index/ ...
------------------------------
Manifest creato con successo: TCGA_MANIFEST_FROM_DIR.csv
Totale slide processate: 17
     patient_id  n_wsi                                          slide_ids  \
0       9774472      1                                        ['9774472']   
1         15236      1                                          ['15236']   
2  TCGA-DM-A28E      1  ['TCGA-DM-A28E-01Z-00-DX1.4381ffe6-3918-4fdd-b...   
3  TCGA-DM-A1D9      1  ['TCGA-DM-A1D9-01Z-00-DX1.C286F663-142A-4F8E-B...   
4  TCGA-AA-A024      1  ['TCGA-AA-A024-01Z-00-DX1.5F24A31C-2F11-4768-9...   

                                          feat_paths  n_tiles_total  \
0  ['/hpcscratch/ieo/ieo7627/features/more_23/977...           2763   
1  ['/hpcscratch/ieo/ieo7627/features/more_23/152...           1301   
2  ['/hpcscratch/ieo/ieo7627/features/more_23/TCG...           3291   
3  ['/hpcscratch/ieo/ieo7627/features/more_23/TCG

In [4]:
import os
import glob
import re
import json
import pandas as pd

# --- CONFIGURAZIONE ---
# Cartella dove sono i file indicizzati (parquet e json)
# Usa il percorso che hai indicato nei log precedenti
TILE_INDEX_DIR = "/hpcscratch/ieo/ieo7627/parquet_TITAN_TCGA/more_23/tile_index/"

# Regex per trovare l'ID Paziente TCGA (es. TCGA-AA-A024)
tcga_pattern = re.compile(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})")

print(f"🔍 Scansione directory: {TILE_INDEX_DIR}")

# Cerchiamo tutti i file parquet
files = glob.glob(os.path.join(TILE_INDEX_DIR, "*.parquet"))

if not files:
    print("❌ Nessun file parquet trovato. Controlla il percorso.")
    exit()

risultati = []

for file_path in files:
    filename = os.path.basename(file_path)
    stem = filename.replace('.parquet', '')
    
    patient_id = "NON_TROVATO"
    metodo = "N/A"
    
    # --- TENTATIVO 1: Cerca nel nome del file (più veloce) ---
    match = tcga_pattern.search(filename)
    if match:
        patient_id = match.group(1)
        metodo = "Filename"
    
    # --- TENTATIVO 2: Se il nome è strano (es. numerico), cerca nel JSON ---
    else:
        # Costruisce il percorso del presunto file json associato
        json_path = file_path.replace('.parquet', '.json')
        
        if os.path.exists(json_path):
            try:
                with open(json_path, 'r') as f:
                    data = json.load(f)
                    # Cerca chiavi comuni dove potrebbe essere il nome originale
                    # Modifica queste chiavi se il tuo json ha struttura diversa
                    raw_name = data.get('filename') or data.get('slide_id') or data.get('name') or ""
                    
                    # Cerca il pattern TCGA nel valore estratto dal JSON
                    match_json = tcga_pattern.search(str(raw_name))
                    if match_json:
                        patient_id = match_json.group(1)
                        metodo = "JSON Metadata"
                    else:
                        metodo = "JSON (Pattern non trovato)"
            except Exception as e:
                metodo = f"Errore lettura JSON: {e}"
        else:
            metodo = "JSON mancante"

    # --- Salvataggio Risultato ---
    risultati.append({
        'filename': filename,
        'patient_id': patient_id,
        'source_method': metodo,
        'full_path_parquet': file_path
    })

# Creazione DataFrame
df_results = pd.DataFrame(risultati)

# Mostra i risultati
print("-" * 50)
print(f"✅ Analizzati {len(df_results)} file.")
print(f"Pazienti identificati: {df_results[df_results['patient_id'] != 'NON_TROVATO'].shape[0]}")
print("-" * 50)

# Mostra i casi "strani" (dove abbiamo dovuto guardare nel JSON o abbiamo fallito)
casi_complessi = df_results[df_results['source_method'] != 'Filename']
if not casi_complessi.empty:
    print("\n⚠️ File con nomi non standard (risolti tramite JSON o falliti):")
    print(casi_complessi[['filename', 'patient_id', 'source_method']])
else:
    print("\nTutti i file avevano l'ID nel nome del file.")

# Salva l'elenco per uso futuro
df_results.to_csv("lista_pazienti_da_parquet.csv", index=False)
print("\n📁 Lista salvata in 'lista_pazienti_da_parquet.csv'")

🔍 Scansione directory: /hpcscratch/ieo/ieo7627/parquet_TITAN_TCGA/more_23/tile_index/
--------------------------------------------------
✅ Analizzati 17 file.
Pazienti identificati: 11
--------------------------------------------------

⚠️ File con nomi non standard (risolti tramite JSON o falliti):
                filename   patient_id  source_method
0        9774472.parquet  NON_TROVATO  JSON mancante
1          15236.parquet  NON_TROVATO  JSON mancante
5   852919548387.parquet  NON_TROVATO  JSON mancante
10      48293191.parquet  NON_TROVATO  JSON mancante
12       3617706.parquet  NON_TROVATO  JSON mancante
14       3328325.parquet  NON_TROVATO  JSON mancante

📁 Lista salvata in 'lista_pazienti_da_parquet.csv'


In [3]:
import pandas as pd
import os
import re

# --- CONFIGURAZIONE ---
INPUT_CSV = '/hpcscratch/ieo/ieo7627/parquet_TITAN_TCGA/more_6/tile_index_manifest.csv'  # Il file prodotto dal tiling
OUTPUT_CSV = 'TCGA_MANIFEST_FINAL_6.csv' # Il nome del nuovo manifest da creare

# Cartella dove finiranno le features (file .pt)
# Assicurati che questo percorso sia corretto per i nuovi pazienti
FEATURES_DIR = "/hpcscratch/ieo/ieo7627/features/more_6/"

# ----------------------

def create_manifest_from_csv():
    # 1. Carica il file del tiling
    if not os.path.exists(INPUT_CSV):
        print(f"Errore: Il file '{INPUT_CSV}' non esiste.")
        return

    df_tiling = pd.read_csv(INPUT_CSV)
    print(f"Letto file {INPUT_CSV} con {len(df_tiling)} righe.")

    manifest_rows = []

    for index, row in df_tiling.iterrows():
        # --- A. Estrazione Dati ---
        filename = row['filename']       # Es: TCGA-AA-A024-01Z...svs
        n_tiles = row['n_qc_ok']         # Usiamo le tile che hanno superato il QC
        
        # --- B. Pulizia Nome File e ID ---
        # Rimuove l'estensione .svs per ottenere lo "stem" (usato per i nomi dei file .pt)
        stem = filename.replace('.svs', '')
        
        # Estrae il Patient ID (i primi 12 caratteri: TCGA-XX-XXXX)
        # Usiamo una regex per essere sicuri di prendere il formato giusto
        match = re.search(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})", filename)
        if match:
            patient_id = match.group(1)
        else:
            print(f"ATTENZIONE: Nessun pattern TCGA trovato in {filename}. Uso i primi 12 caratteri.")
            patient_id = filename[:12]

        # --- C. Costruzione Percorsi ---
        # Costruisce il percorso completo del futuro file .pt
        feat_path = os.path.join(FEATURES_DIR, f"{stem}.pt")

        # --- D. Formattazione Liste (Stringhe) ---
        # Il manifest richiede che questi campi siano stringhe che sembrano liste
        slide_ids_str = f"['{stem}']"
        feat_paths_str = f"['{feat_path}']"

        # --- E. Creazione Riga ---
        entry = {
            'patient_id': patient_id,
            'n_wsi': 1,
            'slide_ids': slide_ids_str,
            'feat_paths': feat_paths_str,
            'n_tiles_total': n_tiles,
            'feature_dim_mode': "768",     # Valore standard
            'all_feat_exist': True         # Valore standard
        }
        manifest_rows.append(entry)

    # 2. Creazione DataFrame Finale
    df_manifest = pd.DataFrame(manifest_rows)

    # 3. Riordina le colonne come nel manifest originale
    columns_order = ['patient_id', 'n_wsi', 'slide_ids', 'feat_paths', 
                     'n_tiles_total', 'feature_dim_mode', 'all_feat_exist']
    
    # Assicurati che esistano tutte le colonne (in caso il df sia vuoto)
    if not df_manifest.empty:
        df_manifest = df_manifest[columns_order]

    # 4. Salva su CSV
    df_manifest.to_csv(OUTPUT_CSV, index=False)
    
    print("-" * 40)
    print(f"✅ Manifest creato: {OUTPUT_CSV}")
    print(f"📊 Totale pazienti: {len(df_manifest)}")
    print("-" * 40)
    if not df_manifest.empty:
        print("Prime 5 righe:")
        print(df_manifest.head())

if __name__ == "__main__":
    create_manifest_from_csv()

Letto file /hpcscratch/ieo/ieo7627/parquet_TITAN_TCGA/more_6/tile_index_manifest.csv con 6 righe.
----------------------------------------
✅ Manifest creato: TCGA_MANIFEST_FINAL_6.csv
📊 Totale pazienti: 6
----------------------------------------
Prime 5 righe:
     patient_id  n_wsi                                          slide_ids  \
0  TCGA-CM-5864      1  ['TCGA-CM-5864-01Z-00-DX1.2cb87875-6cae-4d8e-9...   
1  TCGA-CM-6168      1  ['TCGA-CM-6168-01Z-00-DX1.96af6eb2-9d51-4671-b...   
2  TCGA-5M-AATE      1  ['TCGA-5M-AATE-01Z-00-DX1.483FFD2F-61A1-477E-8...   
3  TCGA-CM-6162      1  ['TCGA-CM-6162-01Z-00-DX1.806a99a3-cda2-4dde-8...   
4  TCGA-CM-6163      1  ['TCGA-CM-6163-01Z-00-DX1.012a7433-73bb-4584-9...   

                                          feat_paths  n_tiles_total  \
0  ['/hpcscratch/ieo/ieo7627/features/more_6/TCGA...           3382   
1  ['/hpcscratch/ieo/ieo7627/features/more_6/TCGA...           3819   
2  ['/hpcscratch/ieo/ieo7627/features/more_6/TCGA...           

In [4]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, message=r".*weights_only=False.*")

import numpy as np, torch, matplotlib.pyplot as plt, pandas as pd
from torch.utils.data import DataLoader
from scipy.special import expit

# ===================== CONFIG INFERENZA =====================
MIL_KIND = "transmil"
SPLIT_SEEDS = [58, 1]
INIT_SEEDS  = [42]
N_FOLDS     = 5

MANIFEST_TEST = "TCGA_MANIFEST_FINAL_6.csv"   # <-- il manifest dei 12 nuovi pazienti
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===================== UTILS =====================
def infer_loader_test(model, loader, device):
    """
    Come infer_fold ma robusto a y mancanti (test set).
    Ritorna y_true (può contenere NaN), y_prob, patient_id.
    """
    model.eval()
    ys, ps, pids = [], [], []
    with torch.no_grad():
        for batch in loader:
            H = batch.get("bag_feats", batch.get("H"))
            if isinstance(H, np.ndarray):
                H = torch.from_numpy(H)
            H = H.to(device).float() if H is not None else None
            if H is None or H.ndim != 2 or H.size(0) == 0:
                H = torch.zeros((1, 768), dtype=torch.float32, device=device)

            out = model(H)
            logit = out["logit_bin"]
            prob = torch.sigmoid(logit).view(-1)[0].item()
            ps.append(float(prob))

            # y può non esserci nel test → mettiamo NaN
            yb = batch.get("y_bin", None)
            if yb is None:
                ys.append(np.nan)
            else:
                y_val = _scalar(yb)
                ys.append(float(y_val))

            pid = batch.get("patient_id")
            if isinstance(pid, (list, tuple)):
                pid = pid[0]
            if isinstance(pid, torch.Tensor):
                pid = pid.item() if pid.ndim == 0 else pid[0].item()
            pids.append(str(pid))

    return np.array(ys), np.array(ps), np.array(pids)


# ===================== DATASET TEST (senza fold) =====================
ds_test = PatientBagsDataset(
    manifest_csv=MANIFEST_TEST,
    norm_mode="slide_zscore",
    sample_per_patient=1024,
)

# usiamo la modalità "val" per avere sampling deterministico.
# fold fittizio = 0: le tile sono sempre le stesse per tutti i modelli.
ds_test.set_split("val", fold=0)

g, wif = make_loader_seed(12345)  # seed fisso per riproducibilità
test_loader = DataLoader(
    ds_test,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    worker_init_fn=wif,
    generator=g,
    collate_fn=collate_patient
)

# ===================== RESCALING STATS (da OOF dei 89) =====================
logit_stats = pd.read_csv("logit_rescaling_stats.csv")

# helper: prendi mu,sigma per un dato (split_seed, fold)
def get_mu_sigma(split_seed, fold):
    row = logit_stats[
        (logit_stats["split_seed"] == split_seed) &
        (logit_stats["fold"] == fold)
    ]
    if row.empty:
        raise ValueError(f"Nessuna stat di rescaling per split_seed={split_seed}, fold={fold}")
    row = row.iloc[0]
    return float(row["logit_mean"]), float(row["logit_std"])


# ===================== LOOP MODELLI: INFERENZA TEST =====================
rows_all = []

for SPLIT_SEED in SPLIT_SEEDS:
    for INIT_SEED in INIT_SEEDS:
        for f in range(N_FOLDS):
            ckpt_path = f"{MIL_KIND}_split{SPLIT_SEED}_init{INIT_SEED}_fold{f}_best_Kbag.pth"
            print(f"\n[TEST] uso checkpoint: {ckpt_path}")

            # build model + carica pesi
            if MIL_KIND == "clam":
                model = build_mil_head("clam", in_dim=768, attn_dim=128, dropout=DROPOUT).to(DEVICE)
            elif MIL_KIND == "dsmil":
                model = build_mil_head("dsmil", in_dim=768, attn_dim=128, dropout=DROPOUT, alpha=0.5).to(DEVICE)
            elif MIL_KIND == "transmil":
                model = build_mil_head("transmil", in_dim=768, d_model=768, n_heads=4, n_layers=2,
                                       dim_ff=3072, dropout=DROPOUT, use_relpos2d=False).to(DEVICE)
            else:
                raise ValueError("MIL_KIND deve essere 'clam' | 'dsmil' | 'transmil'")

            state = torch.load(ckpt_path, map_location=DEVICE)
            model.load_state_dict(state)

            # inferenza sui 12 pazienti
            y_test, p_test_raw, pid_test = infer_loader_test(model, test_loader, DEVICE)

            # rescaling logit come negli OOF
            mu, sigma = get_mu_sigma(SPLIT_SEED, f)
            eps = 1e-6
            p_clip = np.clip(p_test_raw, eps, 1 - eps)
            logit = np.log(p_clip / (1 - p_clip))
            logit_std = (logit - mu) / (sigma + 1e-6)
            p_test_resc = expit(logit_std)

            for yy, pr, prr, pid in zip(y_test, p_test_raw, p_test_resc, pid_test):
                rows_all.append({
                    "patient_id": pid,
                    "y": yy if yy == yy else None,   # NaN → None
                    "p_raw": float(pr),
                    "p_rescaled": float(prr),
                    "split_seed": int(SPLIT_SEED),
                    "init_seed": int(INIT_SEED),
                    "fold": int(f),
                })

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

df_test_all = pd.DataFrame(rows_all)
df_test_all.to_csv(f"{MIL_KIND}_TEXTERNAl_TCGA_6.csv", index=False)
print("\nSalvato:", f"{MIL_KIND}_TEST_allmodels.csv")
display(df_test_all.head())


# ===================== AGGREGAZIONE PER PAZIENTE (ENSEMBLE) =====================
agg_test = (
    df_test_all
    .groupby("patient_id")
    .agg(
        p_raw_mean=("p_raw", "mean"),
        p_resc_mean=("p_rescaled", "mean"),
        n_models=("p_rescaled", "count"),
        y=("y", "first"),
    )
    .reset_index()
)

agg_test.to_csv(f"{MIL_KIND}_External_TCGA_6.csv", index=False)
print("\nPredizioni aggregate per paziente salvate in:", f"{MIL_KIND}_TEST_agg.csv")
display(agg_test.head())



[TEST] uso checkpoint: transmil_split58_init42_fold0_best_Kbag.pth

[TEST] uso checkpoint: transmil_split58_init42_fold1_best_Kbag.pth

[TEST] uso checkpoint: transmil_split58_init42_fold2_best_Kbag.pth

[TEST] uso checkpoint: transmil_split58_init42_fold3_best_Kbag.pth

[TEST] uso checkpoint: transmil_split58_init42_fold4_best_Kbag.pth

[TEST] uso checkpoint: transmil_split1_init42_fold0_best_Kbag.pth

[TEST] uso checkpoint: transmil_split1_init42_fold1_best_Kbag.pth

[TEST] uso checkpoint: transmil_split1_init42_fold2_best_Kbag.pth

[TEST] uso checkpoint: transmil_split1_init42_fold3_best_Kbag.pth

[TEST] uso checkpoint: transmil_split1_init42_fold4_best_Kbag.pth

Salvato: transmil_TEST_allmodels.csv


,patient_id,y,p_raw,p_rescaled,split_seed,init_seed,fold
0,TCGA-CM-5864,None,0.798608,0.524145,58,42,0
1,TCGA-CM-6168,None,0.968981,0.667831,58,42,0
2,TCGA-5M-AATE,None,0.443301,0.408208,58,42,0
3,TCGA-CM-6162,None,0.976816,0.686889,58,42,0
4,TCGA-CM-6163,None,0.944366,0.627284,58,42,0



Predizioni aggregate per paziente salvate in: transmil_TEST_agg.csv


,patient_id,p_raw_mean,p_resc_mean,n_models,y
0,TCGA-5M-AATE,0.438776,0.561008,10,None
1,TCGA-CM-5864,0.661291,0.623737,10,None
2,TCGA-CM-6162,0.851264,0.717463,10,None
3,TCGA-CM-6163,0.646277,0.599850,10,None
4,TCGA-CM-6168,0.743902,0.675245,10,None


In [4]:
import os, pandas as pd, ast

m = pd.read_csv(MANIFEST_TEST)

def parse_list(x):
    if isinstance(x, list): return x
    if pd.isna(x): return []
    s = str(x)
    try:
        return ast.literal_eval(s)
    except:
        return [t.strip() for t in s.split(",") if t.strip()]

m["feat_paths_list"] = m["feat_paths"].apply(parse_list)
m["n_paths"] = m["feat_paths_list"].apply(len)
m["n_exists"] = m["feat_paths_list"].apply(lambda L: sum(os.path.exists(p) for p in L if isinstance(p,str)))
print(m[["patient_id","n_paths","n_exists"]].head(20))
print("rows with n_exists==0:", (m["n_exists"]==0).sum(), "/", len(m))


      patient_id  n_paths  n_exists
0   TCGA-AA-A024        1         0
1   TCGA-AA-A029        1         0
2   TCGA-DM-A28A        1         0
3   TCGA-DM-A285        1         0
4   TCGA-AA-A010        1         0
5   TCGA-AA-A01P        1         0
6   TCGA-DM-A1D9        1         0
7   TCGA-AA-3956        1         0
8   TCGA-AA-A01R        1         0
9   TCGA-CA-6718        1         0
10  TCGA-AA-A01V        1         0
11  TCGA-DM-A280        1         0
12  TCGA-DM-A28E        1         0
13  TCGA-AA-A01X        1         0
14  TCGA-AA-A01S        1         0
15  TCGA-4T-AA8H        1         0
16  TCGA-DM-A1DB        1         0
rows with n_exists==0: 17 / 17


In [7]:
import numpy as np
import pandas as pd
from scipy.special import expit

IN_CSV  = "transmil_TEST_allmodels.csv"           # quello con p_raw
OUT_RUN = "transmil_TEST_allmodels_rescaled_K.csv"  # opzionale: per riga-modello
OUT_AGG = "transmil_TEST_agg_rescaled_K.csv"        # per paziente

# 1) Carica
df = pd.read_csv(IN_CSV).copy()
# mi aspetto almeno: patient_id, y, p_raw, split_seed, fold, (init_seed opzionale)

# 2) Usa p_raw come p
df["p"] = df["p_raw"].astype(float)

# 2.1 calcola il logit
eps = 1e-6
p_clip = np.clip(df["p"].values, eps, 1 - eps)
df["logit"] = np.log(p_clip / (1 - p_clip))

# 2.2 standardizza il logit per (split_seed, fold)
df["logit_std"] = df.groupby(["split_seed", "fold"])["logit"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-6)
)

# 2.3 ricava probabilità "rescaled" per riga
df["p_rescaled"] = expit(df["logit_std"])

# (opzionale) salva il file per-run
df.to_csv(OUT_RUN, index=False)
print("Salvato per-run:", OUT_RUN)

# 2.4 aggrega per paziente usando p_rescaled
agg = (
    df
    .groupby("patient_id")
    .agg(
        p_mean=("p_rescaled", "mean"),
        p_std=("p_rescaled", "std"),
        n_runs=("p_rescaled", "count"),
        y=("y", "first"),
    )
    .reset_index()
)

agg.to_csv(OUT_AGG, index=False)
print("Salvato aggregato:", OUT_AGG)
display(agg.head())


Salvato per-run: transmil_TEST_allmodels_rescaled_K.csv
Salvato aggregato: transmil_TEST_agg_rescaled_K.csv


,patient_id,p_mean,p_std,n_runs,y
0,25-I-06976,0.461297,0.247701,10,0.0
1,25-I-07899,0.459070,0.220522,10,0.0
2,25-I-08119,0.473131,0.143173,10,0.0
3,25-I-09361,0.630673,0.190574,10,1.0
4,25-I-11436,0.329203,0.114794,10,0.0


In [8]:
import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss, confusion_matrix

TEST_ALL = "transmil_TEST_allmodels.csv"   # 15x13 righe: patient_id, y, p_raw, split_seed, fold, ...
OUT_SENS = "test_patient_sensitivity_K.csv"
OUT_BAG  = "test_bagging_comparison_K.csv"

EPS = 1e-6

def logit(p):
    p = np.clip(np.asarray(p, float), EPS, 1 - EPS)
    return np.log(p / (1 - p))

def safe_auc(y, p):
    y = np.asarray(y, int)
    if len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, p)

def metrics(name, y, p, thr=0.5):
    y = np.asarray(y, int)
    p = np.asarray(p, float)
    pred = (p >= thr).astype(int)
    return {
        "name": name,
        "n": int(len(y)),
        "acc@0.5": float(accuracy_score(y, pred)),
        "auc": float(safe_auc(y, p)),
        "brier": float(brier_score_loss(y, p)),
        "tn_fp_fn_tp": tuple(map(int, confusion_matrix(y, pred, labels=[0,1]).ravel()))
    }

# ---------------------------
# Load
# ---------------------------
df = pd.read_csv(TEST_ALL)
df["patient_id"] = df["patient_id"].astype(str).str.strip()

# column detection
prob_raw_col = "p_raw" if "p_raw" in df.columns else ("p" if "p" in df.columns else None)
if prob_raw_col is None:
    raise ValueError(f"Non trovo p_raw/p. Colonne: {df.columns.tolist()}")

df["p_raw"] = df[prob_raw_col].astype(float)
if "y" in df.columns:
    df["y"] = df["y"].astype(int)

for c in ["split_seed", "fold"]:
    if c not in df.columns:
        raise ValueError(f"Manca colonna '{c}' nel TEST_allmodels.")
    df[c] = df[c].astype(int)

# ---------------------------
# (A) Sensitivity decomposition
# ---------------------------
# 1) per-patient overall across all models (15)
overall = df.groupby("patient_id").agg(
    n_models=("p_raw","count"),
    p_mean15=("p_raw","mean"),
    p_median15=("p_raw","median"),
    p_std15=("p_raw","std"),
    p_min15=("p_raw","min"),
    p_max15=("p_raw","max"),
).reset_index()

overall["range15"] = overall["p_max15"] - overall["p_min15"]
overall["logit_mean15"] = df.groupby("patient_id")["p_raw"].apply(lambda x: float(expit(np.mean(logit(x))))).values

votes15 = df.assign(vote=(df["p_raw"]>=0.5).astype(int)).groupby("patient_id")["vote"].mean().reset_index()
votes15.rename(columns={"vote":"frac_pos15"}, inplace=True)
overall = overall.merge(votes15, on="patient_id", how="left")
overall["consensus15"] = np.maximum(overall["frac_pos15"], 1 - overall["frac_pos15"])

if "y" in df.columns:
    overall = overall.merge(df.groupby("patient_id")["y"].first().reset_index(), on="patient_id", how="left")

# 2) per patient x split_seed: aggrego sulle 5 fold
per_seed = df.groupby(["patient_id","split_seed"]).agg(
    p_mean_seed=("p_raw","mean"),
    p_median_seed=("p_raw","median"),
    p_std_within_seed=("p_raw","std"),      # variabilità tra fold dentro lo stesso seed
    p_min_seed=("p_raw","min"),
    p_max_seed=("p_raw","max"),
).reset_index()

per_seed["range_seed"] = per_seed["p_max_seed"] - per_seed["p_min_seed"]
per_seed["logit_mean_seed"] = df.groupby(["patient_id","split_seed"])["p_raw"].apply(
    lambda x: float(expit(np.mean(logit(x))))
).values
per_seed["frac_pos_seed"] = df.groupby(["patient_id","split_seed"])["p_raw"].apply(lambda x: float(np.mean(np.asarray(x)>=0.5))).values
per_seed["consensus_seed"] = np.maximum(per_seed["frac_pos_seed"], 1 - per_seed["frac_pos_seed"])

# 3) collapse seed-level -> patient-level sensitivity
seed_sens = per_seed.groupby("patient_id").agg(
    std_across_seed_means=("p_mean_seed","std"),            # variazione tra i 3 split_seed
    std_across_seed_logitmeans=("logit_mean_seed","std"),
    mean_fold_std=("p_std_within_seed","mean"),             # quanto variano le 5 fold (in media sui seed)
    max_fold_std=("p_std_within_seed","max"),
    mean_range_within_seed=("range_seed","mean"),
    max_range_within_seed=("range_seed","max"),
).reset_index()

sens = overall.merge(seed_sens, on="patient_id", how="left")

# ranking “who’s the culprit?”
# - fold-sensitive: mean_fold_std / max_fold_std alto
# - seed-sensitive: std_across_seed_means alto
sens["rank_fold_sensitive"] = sens["mean_fold_std"].rank(ascending=False, method="min")
sens["rank_seed_sensitive"] = sens["std_across_seed_means"].rank(ascending=False, method="min")

sens.to_csv(OUT_SENS, index=False)
print("Saved:", OUT_SENS)
display(sens.sort_values("mean_fold_std", ascending=False).head(10))

print("\nTop 5 FOLD-sensitive (mean_fold_std):")
display(sens.sort_values("mean_fold_std", ascending=False)[
    ["patient_id","y","p_mean15","p_median15","p_std15","range15","mean_fold_std","max_fold_std","std_across_seed_means","consensus15"]
].head(5))

print("\nTop 5 SEED-sensitive (std_across_seed_means):")
display(sens.sort_values("std_across_seed_means", ascending=False)[
    ["patient_id","y","p_mean15","p_median15","p_std15","range15","std_across_seed_means","mean_fold_std","consensus15"]
].head(5))

# ---------------------------
# (B) Test-time bagging strategies from existing predictions
# ---------------------------
# define aggregators producing one probability per patient
# 1) mean/median over 15
pred_mean15 = overall.set_index("patient_id")["p_mean15"]
pred_med15  = overall.set_index("patient_id")["p_median15"]
pred_logit15= overall.set_index("patient_id")["logit_mean15"]

# 2) seed-balanced: mean over folds inside each seed, then mean over 3 seeds (equal seed weights)
seed_bal_mean = per_seed.groupby("patient_id")["p_mean_seed"].mean()
seed_bal_logit= per_seed.groupby("patient_id")["logit_mean_seed"].mean()

# 3) seed-majority vote: each seed votes on p_mean_seed>=0.5, then average votes -> prob in {0,1/3,2/3,1}
seed_vote = (per_seed.assign(v=(per_seed["p_mean_seed"]>=0.5).astype(int))
                   .groupby("patient_id")["v"].mean())

# 4) trimmed mean over 15 (robust against a few crazy folds/models)
def trimmed_mean(arr, trim=0.2):
    a = np.sort(np.asarray(arr, float))
    k = int(len(a) * trim)
    if 2*k >= len(a):  # fallback
        return float(np.mean(a))
    return float(np.mean(a[k:len(a)-k]))

trim20 = df.groupby("patient_id")["p_raw"].apply(lambda x: trimmed_mean(x, trim=0.2))

bag_df = pd.DataFrame({
    "patient_id": overall["patient_id"].values,
    "p_mean15": overall["p_mean15"].values,
    "p_median15": overall["p_median15"].values,
    "p_logitmean15": overall["logit_mean15"].values,
    "p_seed_bal_mean": seed_bal_mean.reindex(overall["patient_id"]).values,
    "p_seed_bal_logitmean": seed_bal_logit.reindex(overall["patient_id"]).values,
    "p_seed_vote": seed_vote.reindex(overall["patient_id"]).values,
    "p_trim20": trim20.reindex(overall["patient_id"]).values,
})

if "y" in overall.columns:
    bag_df["y"] = overall["y"].values

bag_df.to_csv(OUT_BAG, index=False)
print("Saved:", OUT_BAG)
display(bag_df.head())

# Metrics table (if y exists)
if "y" in bag_df.columns and bag_df["y"].notna().all():
    y = bag_df["y"].astype(int).values
    results = []
    for col in ["p_mean15","p_median15","p_logitmean15","p_seed_bal_mean","p_seed_bal_logitmean","p_seed_vote","p_trim20"]:
        results.append(metrics(col, y, bag_df[col].values, thr=0.5))
    res = pd.DataFrame(results).sort_values(["auc","acc@0.5"], ascending=False)
    display(res)

    print("\nNOTE: tn_fp_fn_tp = (TN, FP, FN, TP)")
else:
    print("\n[INFO] Colonna 'y' non presente o incompleta: salto metriche, ma hai salvato tutte le predizioni aggregate.")


Saved: test_patient_sensitivity_K.csv


,patient_id,n_models,p_mean15,p_median15,p_std15,p_min15,p_max15,range15,logit_mean15,frac_pos15,consensus15,y,std_across_seed_means,std_across_seed_logitmeans,mean_fold_std,max_fold_std,mean_range_within_seed,max_range_within_seed,rank_fold_sensitive,rank_seed_sensitive
1,25-I-07899,10,0.447555,0.434057,0.290196,0.115735,0.936930,0.821195,0.450985,0.4,0.6,0,0.061347,0.060526,0.302114,0.335511,0.776562,0.821195,1.0,6.0
8,25-I-14200,10,0.458098,0.467965,0.284084,0.067963,0.814402,0.746439,0.427565,0.4,0.6,1,0.010180,0.001646,0.298357,0.339709,0.725905,0.737795,2.0,12.0
12,25-I-16963,10,0.451291,0.422829,0.284468,0.040726,0.861187,0.820461,0.419141,0.5,0.5,0,0.020548,0.023549,0.294966,0.356357,0.713286,0.820461,3.0,10.0
9,25-I-14723,10,0.588219,0.598273,0.256515,0.188195,0.969729,0.781535,0.640336,0.6,0.6,1,0.058915,0.099449,0.268052,0.270024,0.662165,0.714033,4.0,7.0
6,25-I-13140,10,0.475865,0.430787,0.251599,0.183605,0.893922,0.710317,0.481689,0.4,0.6,1,0.039244,0.053991,0.265017,0.269343,0.664184,0.710317,5.0,8.0
5,25-I-12353,10,0.484871,0.353900,0.267964,0.184797,0.960960,0.776163,0.523521,0.3,0.7,0,0.114212,0.181575,0.260581,0.329319,0.580868,0.657071,6.0,5.0
2,25-I-08119,10,0.452720,0.398736,0.223168,0.193593,0.890881,0.697288,0.459262,0.3,0.7,0,0.008556,0.028568,0.236135,0.251098,0.586581,0.608620,7.0,13.0
7,25-I-14079,10,0.501123,0.460475,0.217717,0.255296,0.823643,0.568347,0.505584,0.5,0.5,1,0.025390,0.038237,0.228408,0.255844,0.494175,0.550197,8.0,9.0
10,25-I-15205,10,0.678329,0.689496,0.201685,0.406561,0.968057,0.561496,0.728483,0.7,0.7,1,0.020003,0.015332,0.211262,0.240925,0.524451,0.555691,9.0,11.0
3,25-I-09361,10,0.599224,0.626759,0.232043,0.259207,0.952521,0.693314,0.634908,0.7,0.7,1,0.161077,0.187250,0.207133,0.245266,0.507541,0.578100,10.0,2.0



Top 5 FOLD-sensitive (mean_fold_std):


,patient_id,y,p_mean15,p_median15,p_std15,range15,mean_fold_std,max_fold_std,std_across_seed_means,consensus15
1,25-I-07899,0,0.447555,0.434057,0.290196,0.821195,0.302114,0.335511,0.061347,0.6
8,25-I-14200,1,0.458098,0.467965,0.284084,0.746439,0.298357,0.339709,0.010180,0.6
12,25-I-16963,0,0.451291,0.422829,0.284468,0.820461,0.294966,0.356357,0.020548,0.5
9,25-I-14723,1,0.588219,0.598273,0.256515,0.781535,0.268052,0.270024,0.058915,0.6
6,25-I-13140,1,0.475865,0.430787,0.251599,0.710317,0.265017,0.269343,0.039244,0.6



Top 5 SEED-sensitive (std_across_seed_means):


,patient_id,y,p_mean15,p_median15,p_std15,range15,std_across_seed_means,mean_fold_std,consensus15
11,25-I-15576,1,0.355071,0.319809,0.214323,0.695514,0.165152,0.179645,0.8
3,25-I-09361,1,0.599224,0.626759,0.232043,0.693314,0.161077,0.207133,0.7
0,25-I-06976,0,0.435560,0.462150,0.225790,0.668387,0.160877,0.202837,0.5
4,25-I-11436,0,0.331837,0.322437,0.201806,0.690076,0.117161,0.186724,0.8
5,25-I-12353,0,0.484871,0.353900,0.267964,0.776163,0.114212,0.260581,0.7


Saved: test_bagging_comparison_K.csv


,patient_id,p_mean15,p_median15,p_logitmean15,p_seed_bal_mean,p_seed_bal_logitmean,p_seed_vote,p_trim20,y
0,25-I-06976,0.435560,0.462150,0.405399,0.435560,0.412384,0.5,0.452777,0
1,25-I-07899,0.447555,0.434057,0.450985,0.447555,0.451348,0.0,0.401152,0
2,25-I-08119,0.452720,0.398736,0.459262,0.452720,0.459329,0.0,0.400969,0
3,25-I-09361,0.599224,0.626759,0.634908,0.599224,0.624765,0.5,0.611445,1
4,25-I-11436,0.331837,0.322437,0.302393,0.331837,0.309661,0.0,0.307861,0


,name,n,acc@0.5,auc,brier,tn_fp_fn_tp
6,p_trim20,13,0.692308,0.857143,0.210956,"(6, 0, 4, 3)"
0,p_mean15,13,0.769231,0.833333,0.216207,"(6, 0, 3, 4)"
3,p_seed_bal_mean,13,0.769231,0.833333,0.216207,"(6, 0, 3, 4)"
1,p_median15,13,0.692308,0.785714,0.209761,"(6, 0, 4, 3)"
2,p_logitmean15,13,0.692308,0.761905,0.210546,"(5, 1, 3, 4)"
4,p_seed_bal_logitmean,13,0.692308,0.761905,0.210787,"(5, 1, 3, 4)"
5,p_seed_vote,13,0.692308,0.738095,0.250000,"(4, 2, 2, 5)"



NOTE: tn_fp_fn_tp = (TN, FP, FN, TP)


In [4]:
# ===================== TEST A (VERO): stesso checkpoint, 5 bag diverse =====================
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss, confusion_matrix

# ---- CONFIG ----
BAG_FOLDS = [0, 1, 2, 3, 4]     # 5 bag deterministiche diverse
SPLIT_SEED = 58
INIT_SEED  = 42
MODEL_FOLD = 0                 # scegli 1 checkpoint (fold del training)
CKPT_PATH  = f"{MIL_KIND}_split{SPLIT_SEED}_init{INIT_SEED}_fold{MODEL_FOLD}_best_Kbag.pth"

OUTDIR = "bag_sensitivity_out_K"
os.makedirs(OUTDIR, exist_ok=True)

OUT_ROWS = os.path.join(OUTDIR, f"bag_rows_{MIL_KIND}_split{SPLIT_SEED}_init{INIT_SEED}_fold{MODEL_FOLD}.csv")
OUT_SUM  = os.path.join(OUTDIR, f"bag_summary_{MIL_KIND}_split{SPLIT_SEED}_init{INIT_SEED}_fold{MODEL_FOLD}.csv")

print("[TEST A] ckpt:", CKPT_PATH)
print("[TEST A] bag_folds:", BAG_FOLDS)

# ---- helper: build model identico al tuo ----
def build_model_for_inference():
    if MIL_KIND == "clam":
        model = build_mil_head("clam", in_dim=768, attn_dim=128, dropout=DROPOUT).to(DEVICE)
    elif MIL_KIND == "dsmil":
        model = build_mil_head("dsmil", in_dim=768, attn_dim=128, dropout=DROPOUT, alpha=0.5).to(DEVICE)
    elif MIL_KIND == "transmil":
        model = build_mil_head(
            "transmil",
            in_dim=768, d_model=768, n_heads=4, n_layers=2,
            dim_ff=3072, dropout=DROPOUT, use_relpos2d=False
        ).to(DEVICE)
    else:
        raise ValueError("MIL_KIND deve essere 'clam' | 'dsmil' | 'transmil'")
    return model

# ---- helper: load state dict (robusto) ----
def load_state_dict_flexible(model, ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE)
    if isinstance(state, dict):
        # casi comuni
        if "state_dict" in state:
            state = state["state_dict"]
        elif "model" in state:
            state = state["model"]
        elif "model_state_dict" in state:
            state = state["model_state_dict"]
    model.load_state_dict(state, strict=True)
    return model

# ---- loader builder (come nel tuo notebook) ----
def make_test_loader(ds):
    # num_workers=0 => niente randomness dai worker
    return DataLoader(
        ds,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_patient
    )

# ---- metriche ----
def safe_auc(y, p):
    y = np.asarray(y, int)
    if len(np.unique(y[~np.isnan(y)])) < 2:
        return np.nan
    return roc_auc_score(y, p)

def metrics(y, p, thr=0.5):
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    m = ~np.isnan(y)
    y = y[m].astype(int); p = p[m]
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {
        "n": int(len(y)),
        "acc@0.5": float(accuracy_score(y, pred)),
        "auc": float(safe_auc(y, p)),
        "brier": float(brier_score_loss(y, p)),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }

# ---- run ----
model = build_model_for_inference()
model = load_state_dict_flexible(model, CKPT_PATH)
model.eval()

rows = []

# IMPORTANT: qui cambiamo SOLO il fold di ds_test in modalità val, per ottenere bag deterministiche diverse
# e ricostruiamo il DataLoader ogni volta.
for bag_fold in BAG_FOLDS:
    ds_test = PatientBagsDataset(
    manifest_csv=MANIFEST_TEST,
    norm_mode="slide_zscore",
    sample_per_patient=1024,
)
    ds_test.set_split("val", fold=bag_fold)

    ds_test.val_idx_cache = {}   # <-- QUESTA è la chiave: forza ricalcolo bag

    test_loader = make_test_loader(ds_test)
    ys, ps, pids = infer_loader_test(model, test_loader, DEVICE)
    for y_i, p_i, pid_i in zip(ys, ps, pids):
        rows.append({
            "patient_id": str(pid_i),
            "y": float(y_i) if y_i is not None else np.nan,
            "bag_fold": int(bag_fold),
            "p_raw": float(p_i),
            "ckpt": CKPT_PATH
        })

# ripristina fold=0 (comodo per non “sporcare” lo stato)
ds_test.set_split("val", fold=0)

df_rows = pd.DataFrame(rows)
df_rows.to_csv(OUT_ROWS, index=False)
print("Saved:", OUT_ROWS)
display(df_rows.head())

# ---- summary per paziente: mean/std/min/max across bag folds ----
g = df_rows.groupby("patient_id").agg(
    y=("y","first"),
    n_bags=("p_raw","count"),
    p_mean=("p_raw","mean"),
    p_std=("p_raw","std"),
    p_min=("p_raw","min"),
    p_max=("p_raw","max"),
).reset_index()
g["range"] = g["p_max"] - g["p_min"]

# ranking: i più bag-sensitive
g = g.sort_values(["p_std","range"], ascending=False).reset_index(drop=True)
g.to_csv(OUT_SUM, index=False)
print("Saved:", OUT_SUM)
display(g.head(10))

# ---- metriche su bag-mean (se y esiste) ----
if g["y"].notna().all():
    print("\n[METRICS] su p_mean (bag-averaged):")
    print(metrics(g["y"].values, g["p_mean"].values, thr=0.5))
else:
    print("\n[INFO] y mancante su alcuni pazienti: salto metriche.")


[TEST A] ckpt: transmil_split58_init42_fold0_best_Kbag.pth
[TEST A] bag_folds: [0, 1, 2, 3, 4]


NameError: name 'MANIFEST_TEST' is not defined

In [9]:
# ===================== TEST B (VERO): 15 checkpoint x 5 bag (basato sul tuo codice) =====================
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from scipy.special import expit
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss, confusion_matrix

# ---- CONFIG ----
BAG_FOLDS   = [0,1,2,3,4]          # 5 bag deterministiche (ATTENZIONE: cache reset dentro loop)
SPLIT_SEEDS = [58, 1]
INIT_SEED   = 42
MODEL_FOLDS = [0,1,2,3,4]          # 5 fold -> 15 checkpoint totali

OUTDIR = "bagging_15x5_out_K"
os.makedirs(OUTDIR, exist_ok=True)

OUT_ROWS = os.path.join(OUTDIR, f"test_rows_{MIL_KIND}_15models_5bags.csv")
OUT_MODEL = os.path.join(OUTDIR, f"test_per_model_{MIL_KIND}_15models.csv")        # per-checkpoint, bag-aggregated
OUT_PAT = os.path.join(OUTDIR, f"test_per_patient_{MIL_KIND}_ensembles.csv")       # per-paziente, ensemble finale

print("[TEST B] MIL_KIND:", MIL_KIND)
print("[TEST B] split_seeds:", SPLIT_SEEDS, "init_seed:", INIT_SEED, "model_folds:", MODEL_FOLDS, "bag_folds:", BAG_FOLDS)

# ---- helper: build model identico al tuo ----
def build_model_for_inference():
    if MIL_KIND == "clam":
        model = build_mil_head("clam", in_dim=768, attn_dim=128, dropout=DROPOUT).to(DEVICE)
    elif MIL_KIND == "dsmil":
        model = build_mil_head("dsmil", in_dim=768, attn_dim=128, dropout=DROPOUT, alpha=0.5).to(DEVICE)
    elif MIL_KIND == "transmil":
        model = build_mil_head(
            "transmil",
            in_dim=768, d_model=768, n_heads=4, n_layers=2,
            dim_ff=3072, dropout=DROPOUT, use_relpos2d=False
        ).to(DEVICE)
    else:
        raise ValueError("MIL_KIND deve essere 'clam' | 'dsmil' | 'transmil'")
    return model

# ---- helper: load state dict (robusto) ----
def load_state_dict_flexible(model, ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE)
    if isinstance(state, dict):
        if "state_dict" in state:
            state = state["state_dict"]
        elif "model" in state:
            state = state["model"]
        elif "model_state_dict" in state:
            state = state["model_state_dict"]
    model.load_state_dict(state, strict=True)
    return model

def make_test_loader(ds):
    return DataLoader(ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_patient)

# ---- math helpers ----
EPS = 1e-6
def logit(p):
    p = np.clip(np.asarray(p, float), EPS, 1-EPS)
    return np.log(p/(1-p))

def safe_auc(y, p):
    y = np.asarray(y, int)
    if len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, p)

def metrics(name, y, p, thr=0.5):
    y = np.asarray(y, int); p = np.asarray(p, float)
    pred = (p>=thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {
        "name": name,
        "n": int(len(y)),
        "acc@0.5": float(accuracy_score(y, pred)),
        "auc": float(safe_auc(y, p)),
        "brier": float(brier_score_loss(y, p)),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }

# ---- RUN ALL ----
rows = []

# fissiamo ds_test in modalità val UNA VOLTA (poi cambiamo fold per la bag)
# NB: il tuo ds_test è già creato nel notebook; qui lo ri-usi.
ds_test.set_split("val", fold=0)

# carichiamo una volta i pid/y “di riferimento” per coerenza
base_loader = make_test_loader(ds_test)
base_y, base_p, base_pid = infer_loader_test(build_model_for_inference().eval(), base_loader, DEVICE)  # dummy, ma serve solo pid/y
# NB: se questo dummy ti dà fastidio, commenta e usa pid/y dal primo modello vero.

# loop checkpoint
for split_seed in SPLIT_SEEDS:
    for model_fold in MODEL_FOLDS:
        ckpt_path = f"{MIL_KIND}_split{split_seed}_init{INIT_SEED}_fold{model_fold}_best_Kbag.pth"
        if not os.path.exists(ckpt_path):
            print("[WARN] Missing:", ckpt_path)
            continue

        model = build_model_for_inference()
        model = load_state_dict_flexible(model, ckpt_path)
        model.eval()

        for bag_fold in BAG_FOLDS:
            ds_test.set_split("val", fold=bag_fold)
            ds_test.val_idx_cache = {}   # FORZA ricalcolo bag (fix fondamentale)

            test_loader = make_test_loader(ds_test)
            ys, ps, pids = infer_loader_test(model, test_loader, DEVICE)

            for y_i, p_i, pid_i in zip(ys, ps, pids):
                rows.append({
                    "patient_id": str(pid_i),
                    "y": float(y_i) if y_i is not None else np.nan,
                    "split_seed": int(split_seed),
                    "model_fold": int(model_fold),
                    "bag_fold": int(bag_fold),
                    "p_raw": float(p_i),
                    "ckpt": ckpt_path
                })

# ripristina fold=0
ds_test.set_split("val", fold=0)

df_rows = pd.DataFrame(rows)
df_rows.to_csv(OUT_ROWS, index=False)
print("Saved:", OUT_ROWS, "shape:", df_rows.shape)
display(df_rows.head())

# ---- 1) Per-checkpoint: aggrega sulle 5 bag (mean/median/logitmean) ----
per_model = df_rows.groupby(["patient_id","split_seed","model_fold"]).agg(
    y=("y","first"),
    p_mean_bag=("p_raw","mean"),
    p_median_bag=("p_raw","median"),
    p_logitmean_bag=("p_raw", lambda x: float(expit(np.mean(logit(x))))),
    p_std_bag=("p_raw","std"),
    p_min_bag=("p_raw","min"),
    p_max_bag=("p_raw","max"),
).reset_index()
per_model["bag_range"] = per_model["p_max_bag"] - per_model["p_min_bag"]

per_model.to_csv(OUT_MODEL, index=False)
print("Saved:", OUT_MODEL, "shape:", per_model.shape)

# ---- 2) Per-paziente: ensemble finale su 15 modelli (usando p_logitmean_bag di default) ----
# Strategie ensemble: mean / median / logit-mean sui valori già bag-aggregated
def patient_ensemble(df_pm, col):
    # df_pm: righe = modelli per paziente
    g = df_pm.groupby("patient_id")[col].agg(["mean","median"]).reset_index()
    # logit-mean a livello modelli
    logitmean = df_pm.groupby("patient_id")[col].apply(lambda x: float(expit(np.mean(logit(x))))).reset_index(name="logitmean")
    g = g.merge(logitmean, on="patient_id", how="left")
    return g

ens = patient_ensemble(per_model, "p_logitmean_bag")
ens.rename(columns={"mean":"ens_mean", "median":"ens_median", "logitmean":"ens_logitmean"}, inplace=True)

# aggiungo y e diagnostiche varianza bag vs modello
y_map = per_model.groupby("patient_id")["y"].first().reset_index()
ens = ens.merge(y_map, on="patient_id", how="left")

# varianza dovuta al modello (dopo bagging): std tra modelli di p_logitmean_bag
model_std = per_model.groupby("patient_id")["p_logitmean_bag"].std().reset_index(name="std_due_to_model")
# varianza dovuta alla bag (media sulle std bag per modello)
bag_std_mean = per_model.groupby("patient_id")["p_std_bag"].mean().reset_index(name="mean_std_due_to_bag")
bag_range_mean = per_model.groupby("patient_id")["bag_range"].mean().reset_index(name="mean_range_due_to_bag")

ens = ens.merge(model_std, on="patient_id", how="left")
ens = ens.merge(bag_std_mean, on="patient_id", how="left")
ens = ens.merge(bag_range_mean, on="patient_id", how="left")

ens.to_csv(OUT_PAT, index=False)
print("Saved:", OUT_PAT, "shape:", ens.shape)
display(ens.sort_values("std_due_to_model", ascending=False).head(10))

# ---- 3) Metriche finali (se y completa) ----
if ens["y"].notna().all():
    y = ens["y"].astype(int).values

    res = []
    for col in ["ens_mean","ens_median","ens_logitmean"]:
        res.append(metrics(f"ensemble({col})", y, ens[col].values, thr=0.5))

    res_df = pd.DataFrame(res).sort_values(["auc","acc@0.5"], ascending=False)
    display(res_df)

    print("\n[DIAGNOSTICA] median std_due_to_model:", float(np.median(ens["std_due_to_model"].fillna(0))))
    print("[DIAGNOSTICA] median mean_std_due_to_bag:", float(np.median(ens["mean_std_due_to_bag"].fillna(0))))
else:
    print("[INFO] y mancante: salto metriche finali.")


[TEST B] MIL_KIND: transmil
[TEST B] split_seeds: [58, 1] init_seed: 42 model_folds: [0, 1, 2, 3, 4] bag_folds: [0, 1, 2, 3, 4]
Saved: bagging_15x5_out_K/test_rows_transmil_15models_5bags.csv shape: (650, 7)


,patient_id,y,split_seed,model_fold,bag_fold,p_raw,ckpt
0,25-I-06976,0.0,58,0,0,0.410703,transmil_split58_init42_fold0_best_Kbag.pth
1,25-I-07899,0.0,58,0,0,0.936930,transmil_split58_init42_fold0_best_Kbag.pth
2,25-I-08119,0.0,58,0,0,0.890881,transmil_split58_init42_fold0_best_Kbag.pth
3,25-I-09361,1.0,58,0,0,0.434092,transmil_split58_init42_fold0_best_Kbag.pth
4,25-I-11436,0.0,58,0,0,0.216092,transmil_split58_init42_fold0_best_Kbag.pth


Saved: bagging_15x5_out_K/test_per_model_transmil_15models.csv shape: (130, 11)
Saved: bagging_15x5_out_K/test_per_patient_transmil_ensembles.csv shape: (13, 8)


,patient_id,ens_mean,ens_median,ens_logitmean,y,std_due_to_model,mean_std_due_to_bag,mean_range_due_to_bag
12,25-I-16963,0.480090,0.446409,0.451570,0.0,0.275735,0.108420,0.272330
1,25-I-07899,0.512540,0.453323,0.585559,0.0,0.273565,0.138424,0.327646
4,25-I-11436,0.423684,0.467489,0.389543,0.0,0.263061,0.129806,0.322512
5,25-I-12353,0.556072,0.501940,0.588795,0.0,0.253045,0.140220,0.323814
6,25-I-13140,0.546419,0.613301,0.560675,1.0,0.244638,0.122716,0.305248
8,25-I-14200,0.392749,0.338713,0.372687,1.0,0.229202,0.144006,0.349988
0,25-I-06976,0.484048,0.458925,0.478452,0.0,0.224507,0.125564,0.313303
3,25-I-09361,0.641474,0.684261,0.680971,1.0,0.221834,0.147683,0.372026
9,25-I-14723,0.545463,0.492277,0.575958,1.0,0.213278,0.136887,0.342484
11,25-I-15576,0.344971,0.247782,0.325113,1.0,0.212346,0.169789,0.406665


,name,n,acc@0.5,auc,brier,TN,FP,FN,TP
1,ensemble(ens_median),13,0.538462,0.619048,0.250833,4,2,4,3
0,ensemble(ens_mean),13,0.538462,0.547619,0.248479,3,3,3,4
2,ensemble(ens_logitmean),13,0.538462,0.523810,0.251756,3,3,3,4



[DIAGNOSTICA] median std_due_to_model: 0.2245067806724466
[DIAGNOSTICA] median mean_std_due_to_bag: 0.1384239596705602


In [11]:
# ===================== VAL ROBUSTA K-BAG per ranking checkpoint =====================
import os, glob
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from scipy.special import expit
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, accuracy_score, confusion_matrix

# ------------------ INPUTS (adatta se serve) ------------------
# Dataset + indici di validazione del fold corrente
# (nel tuo training loop li hai: ds_val e va_idx)
DS_VAL = ds_val
VAL_IDX = va_idx

# Dove cercare i checkpoint (pattern flessibile)
# Esempi:
#   CKPT_GLOB = "transmil_split58_init42_fold0_epoch*.pth"
#   CKPT_GLOB = "transmil_split58_init42_fold0_*.pth"
#   CKPT_GLOB = "transmil_split58_init42_fold0_best.pth"
CKPT_GLOB = f"{MIL_KIND}_split*_init*_fold*_*.pth"  # default: prende tutto ciò che matcha

# Quante bag deterministiche usare in validazione
BAG_FOLDS = [0,1,2,3,4]   # K=5

# Output
OUTDIR = "val_kbag_rankings"
os.makedirs(OUTDIR, exist_ok=True)
OUT_CSV = os.path.join(OUTDIR, f"val_rank_{MIL_KIND}_K{len(BAG_FOLDS)}.csv")

# ------------------ helpers ------------------
EPS = 1e-6
def logit(p):
    p = np.clip(np.asarray(p, float), EPS, 1-EPS)
    return np.log(p/(1-p))

def safe_auc(y, p):
    y = np.asarray(y, int); p = np.asarray(p, float)
    if len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, p)

def metrics_binary(y, p, thr=0.5):
    y = np.asarray(y, int); p = np.asarray(p, float)
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {
        "auc": float(safe_auc(y, p)),
        "pr_auc": float(average_precision_score(y, p)) if len(np.unique(y))==2 else np.nan,
        "brier": float(brier_score_loss(y, p)),
        "acc@0.5": float(accuracy_score(y, pred)),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }

def build_model_for_inference():
    if MIL_KIND == "clam":
        model = build_mil_head("clam", in_dim=768, attn_dim=128, dropout=DROPOUT).to(DEVICE)
    elif MIL_KIND == "dsmil":
        model = build_mil_head("dsmil", in_dim=768, attn_dim=128, dropout=DROPOUT, alpha=0.5).to(DEVICE)
    elif MIL_KIND == "transmil":
        model = build_mil_head(
            "transmil",
            in_dim=768, d_model=768, n_heads=4, n_layers=2,
            dim_ff=3072, dropout=DROPOUT, use_relpos2d=False
        ).to(DEVICE)
    else:
        raise ValueError("MIL_KIND deve essere 'clam' | 'dsmil' | 'transmil'")
    return model

def load_state_dict_flexible(model, ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE)
    if isinstance(state, dict):
        if "state_dict" in state: state = state["state_dict"]
        elif "model" in state: state = state["model"]
        elif "model_state_dict" in state: state = state["model_state_dict"]
    model.load_state_dict(state, strict=True)
    return model

def make_val_loader(ds, idx):
    return DataLoader(
        Subset(ds, idx),
        batch_size=1,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_patient
    )

# fallback se non esiste infer_loader_test
def infer_loader_fallback(model, loader, device):
    model.eval()
    ys, ps, pids = [], [], []
    with torch.no_grad():
        for batch in loader:
            # nel tuo codice collate_patient di solito ritorna: (X, y, pid, ...)
            # qui faccio un parsing robusto
            if isinstance(batch, (list, tuple)):
                X = batch[0]
                y = batch[1] if len(batch) > 1 else None
                pid = batch[2] if len(batch) > 2 else None
            else:
                raise ValueError("Batch format non supportato: modifica infer_loader_fallback in base al tuo collate.")
            X = X.to(device)
            out = model(X)
            # out può essere prob o logit a seconda del tuo head; nel tuo notebook infer_loader_test già gestisce tutto.
            # qui assumo out = prob
            p = float(out.detach().cpu().view(-1)[0])
            ps.append(p)
            ys.append(float(y) if y is not None else np.nan)
            pids.append(str(pid))
    return ys, ps, pids

# scegli la funzione di inferenza corretta
infer_fn = infer_loader_test if "infer_loader_test" in globals() else infer_loader_fallback

# ------------------ core: eval checkpoint con K bag ------------------
def eval_ckpt_kbag(ckpt_path, ds_val, val_idx, bag_folds):
    model = build_model_for_inference()
    model = load_state_dict_flexible(model, ckpt_path)
    model.eval()

    all_rows = []
    for bag_fold in bag_folds:
        ds_val.set_split("val", fold=bag_fold)

        # FIX fondamentale: se hai la cache val_idx_cache, la resetti per cambiare davvero bag
        if hasattr(ds_val, "val_idx_cache"):
            ds_val.val_idx_cache = {}

        loader = make_val_loader(ds_val, val_idx)
        ys, ps, pids = infer_fn(model, loader, DEVICE)

        for y_i, p_i, pid_i in zip(ys, ps, pids):
            all_rows.append({
                "patient_id": str(pid_i),
                "y": int(y_i) if not np.isnan(y_i) else np.nan,
                "bag_fold": int(bag_fold),
                "p_raw": float(p_i),
            })

    df = pd.DataFrame(all_rows)

    # aggrega K bag per paziente: logit-mean (robusto)
    agg = df.groupby("patient_id").agg(
        y=("y","first"),
        p_mean=("p_raw","mean"),
        p_median=("p_raw","median"),
        p_logitmean=("p_raw", lambda x: float(expit(np.mean(logit(x))))),
        p_std=("p_raw","std"),
        p_min=("p_raw","min"),
        p_max=("p_raw","max"),
    ).reset_index()
    agg["range"] = agg["p_max"] - agg["p_min"]

    # metriche su p_logitmean (default)
    y = agg["y"].astype(int).values
    m_logit = metrics_binary(y, agg["p_logitmean"].values, thr=0.5)

    # anche confronto rapido su mean e median (utile)
    m_mean = metrics_binary(y, agg["p_mean"].values, thr=0.5)
    m_med  = metrics_binary(y, agg["p_median"].values, thr=0.5)

    out = {
        "ckpt": ckpt_path,
        "K": len(bag_folds),
        "val_n": int(len(agg)),
        # default score = logitmean
        **{f"logit_{k}": v for k,v in m_logit.items()},
        **{f"mean_{k}": v for k,v in m_mean.items()},
        **{f"median_{k}": v for k,v in m_med.items()},
        # diagnostica sampling
        "median_p_std_across_bags": float(np.median(agg["p_std"].fillna(0).values)),
        "max_p_std_across_bags": float(np.max(agg["p_std"].fillna(0).values)),
        "median_range_across_bags": float(np.median(agg["range"].values)),
        "max_range_across_bags": float(np.max(agg["range"].values)),
    }
    return out

# ------------------ RUN ranking ------------------
ckpts = sorted(glob.glob(CKPT_GLOB))
if len(ckpts) == 0:
    raise FileNotFoundError(f"Nessun checkpoint matcha CKPT_GLOB='{CKPT_GLOB}'. Sei nella cartella giusta?")

print("[VAL-KBAG] Found checkpoints:", len(ckpts))
print("  first:", ckpts[0])
print("  last :", ckpts[-1])

results = []
for i, ckpt in enumerate(ckpts):
    try:
        r = eval_ckpt_kbag(ckpt, DS_VAL, VAL_IDX, BAG_FOLDS)
        results.append(r)
        print(f"[{i+1}/{len(ckpts)}] done:", os.path.basename(ckpt),
              " logit_auc=", f"{r['logit_auc']:.3f}", " logit_brier=", f"{r['logit_brier']:.3f}")
    except Exception as e:
        print("[ERROR] ckpt:", ckpt, "->", repr(e))

df_rank = pd.DataFrame(results)

# ranking: massimizza AUC, poi minimizza Brier
df_rank = df_rank.sort_values(["logit_auc","logit_brier"], ascending=[False, True]).reset_index(drop=True)

df_rank.to_csv(OUT_CSV, index=False)
print("Saved ranking:", OUT_CSV)

display(df_rank.head(20))


NameError: name 'ds_val' is not defined

In [10]:
# ===================== TEST robusto: 15 checkpoint x K bag su ds_test =====================
import os, glob
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from scipy.special import expit
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss, confusion_matrix

BAG_FOLDS   = [0,1,2,3,4]
SPLIT_SEEDS = [58,1]
INIT_SEED   = 42
MODEL_FOLDS = [0,1,2,3,4]

OUTDIR = "test_robust_out"
os.makedirs(OUTDIR, exist_ok=True)
OUT_ROWS = os.path.join(OUTDIR, f"test_rows_{MIL_KIND}_15models_{len(BAG_FOLDS)}bags_K.csv")
OUT_PAT  = os.path.join(OUTDIR, f"test_per_patient_{MIL_KIND}_ensembles_K.csv")

EPS=1e-6
def logit(p):
    p=np.clip(np.asarray(p,float), EPS, 1-EPS)
    return np.log(p/(1-p))

def build_model_for_inference():
    if MIL_KIND == "transmil":
        return build_mil_head("transmil", in_dim=768, d_model=768, n_heads=4, n_layers=2,
                              dim_ff=3072, dropout=DROPOUT, use_relpos2d=False).to(DEVICE)
    elif MIL_KIND == "clam":
        return build_mil_head("clam", in_dim=768, attn_dim=128, dropout=DROPOUT).to(DEVICE)
    elif MIL_KIND == "dsmil":
        return build_mil_head("dsmil", in_dim=768, attn_dim=128, dropout=DROPOUT, alpha=0.5).to(DEVICE)
    else:
        raise ValueError("MIL_KIND must be transmil|clam|dsmil")

def load_state_dict_flexible(model, ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE)
    if isinstance(state, dict):
        for k in ["state_dict","model","model_state_dict"]:
            if k in state:
                state = state[k]; break
    model.load_state_dict(state, strict=True)
    return model

def make_loader(ds):
    return DataLoader(ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_patient)

def metrics(y, p, thr=0.5):
    y=np.asarray(y,int); p=np.asarray(p,float)
    pred=(p>=thr).astype(int)
    tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
    auc = roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan
    return dict(acc=float(accuracy_score(y,pred)), auc=float(auc),
                brier=float(brier_score_loss(y,p)),
                TN=int(tn),FP=int(fp),FN=int(fn),TP=int(tp))

rows=[]

for split_seed in SPLIT_SEEDS:
    for model_fold in MODEL_FOLDS:
        ckpt = f"{MIL_KIND}_split{split_seed}_init{INIT_SEED}_fold{model_fold}_best_Kbag.pth"
        if not os.path.exists(ckpt):
            print("[WARN missing]", ckpt); continue

        model = load_state_dict_flexible(build_model_for_inference(), ckpt)
        model.eval()

        for bag_fold in BAG_FOLDS:
            ds_test.set_split("val", fold=bag_fold)
            if hasattr(ds_test, "val_idx_cache"):
                ds_test.val_idx_cache = {}   # fondamentale
            loader = make_loader(ds_test)
            ys, ps, pids = infer_loader_test(model, loader, DEVICE)

            for y_i, p_i, pid_i in zip(ys, ps, pids):
                rows.append(dict(patient_id=str(pid_i), y=int(y_i),
                                 split_seed=int(split_seed), model_fold=int(model_fold),
                                 bag_fold=int(bag_fold), p_raw=float(p_i)))

df_rows = pd.DataFrame(rows)
df_rows.to_csv(OUT_ROWS, index=False)
print("Saved:", OUT_ROWS, "shape:", df_rows.shape)

# per modello: aggrega K bag -> p_model
per_model = df_rows.groupby(["patient_id","split_seed","model_fold"]).agg(
    y=("y","first"),
    p_model=("p_raw", lambda x: float(expit(np.mean(logit(x))))),  # logit-mean sulle bag
    p_std_bag=("p_raw","std"),
).reset_index()

# per paziente: ensemble su 15 modelli
ens = per_model.groupby("patient_id").agg(
    y=("y","first"),
    p_final_mean=("p_model","mean"),
    p_final_median=("p_model","median"),
    p_final_logitmean=("p_model", lambda x: float(expit(np.mean(logit(x))))),
    std_due_to_model=("p_model","std"),
    mean_std_due_to_bag=("p_std_bag","mean"),
).reset_index()

ens.to_csv(OUT_PAT, index=False)
print("Saved:", OUT_PAT, "shape:", ens.shape)

y = ens["y"].astype(int).values
print("mean   :", metrics(y, ens["p_final_mean"].values))
print("median :", metrics(y, ens["p_final_median"].values))
print("logit  :", metrics(y, ens["p_final_logitmean"].values))


Saved: test_robust_out/test_rows_transmil_15models_5bags_K.csv shape: (650, 6)
Saved: test_robust_out/test_per_patient_transmil_ensembles_K.csv shape: (13, 7)
mean   : {'acc': 0.5384615384615384, 'auc': 0.5476190476190477, 'brier': 0.24820771471756797, 'TN': 3, 'FP': 3, 'FN': 3, 'TP': 4}
median : {'acc': 0.5384615384615384, 'auc': 0.6190476190476191, 'brier': 0.2508333521197057, 'TN': 4, 'FP': 2, 'FN': 4, 'TP': 3}
logit  : {'acc': 0.5384615384615384, 'auc': 0.5238095238095238, 'brier': 0.2513077800365044, 'TN': 3, 'FP': 3, 'FN': 3, 'TP': 4}


In [ ]:
# ===================== OOF 3(modelli) x 5(bag) FAIR + confronto vs TEST =====================
import os, numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader, Subset
from scipy.special import expit
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss, confusion_matrix

# ---------- CONFIG ----------
OOF_CSV = "transmil_OOF_allseeds_TITAN_reg (10).csv"   # tuo OOF
TEST_ROWS_CSV = "test_rows_transmil_15models_5bags (1).csv"  # opzionale: metti None se non vuoi confronto

SPLIT_SEEDS = [58, 1, 42]
INIT_SEED = 42
MODEL_FOLDS = [0,1,2,3,4]     # fold di training (5 checkpoint per split_seed)
BAG_FOLDS = [0,1,2,3,4]       # 5 bag deterministiche diverse (come hai fatto sul test)

OUTDIR = "oof_3models_5bags_fair_out"
os.makedirs(OUTDIR, exist_ok=True)
OUT_ROWS = os.path.join(OUTDIR, f"oof_rows_{MIL_KIND}_3models_{len(BAG_FOLDS)}bags.csv")
OUT_PAT  = os.path.join(OUTDIR, f"oof_per_patient_{MIL_KIND}_3models.csv")
OUT_SUM  = os.path.join(OUTDIR, f"oof_vs_test_summary_{MIL_KIND}.csv")

DEVICE = DEVICE  # già nel tuo notebook

# ---------- helpers ----------
EPS = 1e-6
def logit(p):
    p = np.clip(np.asarray(p, float), EPS, 1-EPS)
    return np.log(p/(1-p))

def logitmean(arr):
    return float(expit(np.mean(logit(arr))))

def safe_auc(y, p):
    y = np.asarray(y, int); p = np.asarray(p, float)
    if len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, p)

def metrics(y, p, thr=0.5):
    y = np.asarray(y, int); p = np.asarray(p, float)
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {
        "n": int(len(y)),
        "acc@0.5": float(accuracy_score(y, pred)),
        "auc": float(safe_auc(y, p)),
        "brier": float(brier_score_loss(y, p)),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "pred_pos_rate": float(pred.mean()),
        "p_mean": float(np.mean(p)),
        "p_std": float(np.std(p)),
    }

def build_model_for_inference():
    if MIL_KIND == "clam":
        model = build_mil_head("clam", in_dim=768, attn_dim=128, dropout=DROPOUT).to(DEVICE)
    elif MIL_KIND == "dsmil":
        model = build_mil_head("dsmil", in_dim=768, attn_dim=128, dropout=DROPOUT, alpha=0.5).to(DEVICE)
    elif MIL_KIND == "transmil":
        model = build_mil_head(
            "transmil",
            in_dim=768, d_model=768, n_heads=4, n_layers=2,
            dim_ff=3072, dropout=DROPOUT, use_relpos2d=False
        ).to(DEVICE)
    else:
        raise ValueError("MIL_KIND deve essere 'clam' | 'dsmil' | 'transmil'")
    return model

def load_state_dict_flexible(model, ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE)
    if isinstance(state, dict):
        for k in ["state_dict","model","model_state_dict"]:
            if k in state:
                state = state[k]
                break
    model.load_state_dict(state, strict=True)
    return model

def make_loader(ds, idxs):
    return DataLoader(
        Subset(ds, idxs),
        batch_size=1,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_patient
    )
def infer_loader_test(model, loader, device):
    """
    Come infer_fold ma robusto a y mancanti (test set).
    Ritorna y_true (può contenere NaN), y_prob, patient_id.
    """
    model.eval()
    ys, ps, pids = [], [], []
    with torch.no_grad():
        for batch in loader:
            H = batch.get("bag_feats", batch.get("H"))
            if isinstance(H, np.ndarray):
                H = torch.from_numpy(H)
            H = H.to(device).float() if H is not None else None
            if H is None or H.ndim != 2 or H.size(0) == 0:
                H = torch.zeros((1, 768), dtype=torch.float32, device=device)

            out = model(H)
            logit = out["logit_bin"]
            prob = torch.sigmoid(logit).view(-1)[0].item()
            ps.append(float(prob))

            # y può non esserci nel test → mettiamo NaN
            yb = batch.get("y_bin", None)
            if yb is None:
                ys.append(np.nan)
            else:
                y_val = _scalar(yb)
                ys.append(float(y_val))

            pid = batch.get("patient_id")
            if isinstance(pid, (list, tuple)):
                pid = pid[0]
            if isinstance(pid, torch.Tensor):
                pid = pid.item() if pid.ndim == 0 else pid[0].item()
            pids.append(str(pid))

    return np.array(ys), np.array(ps), np.array(pids)

# ---------- IMPORTANT: serve una mappa patient_id -> indice nel dataset ----------
# Il tuo PatientBagsDataset di solito ha o ds.df (manifest) oppure ds.manifest_df o simili.
def get_patient_ids_from_dataset(ds):
    # prova vari attributi comuni
    for attr in ["patient_ids", "pids", "patients"]:
        if hasattr(ds, attr):
            v = getattr(ds, attr)
            try:
                return [str(x) for x in v]
            except:
                pass
    # prova dataframe interno
    for attr in ["df", "manifest_df", "df_manifest", "manifest"]:
        if hasattr(ds, attr):
            d = getattr(ds, attr)
            if isinstance(d, pd.DataFrame) and ("patient_id" in d.columns):
                return d["patient_id"].astype(str).tolist()
    raise RuntimeError("Non riesco a trovare la lista patient_id nel dataset. Aggiusta get_patient_ids_from_dataset() in base alla tua classe.")

# ---------- carica OOF csv e costruisci mapping patient -> fold per split_seed ----------
df_oof = pd.read_csv(OOF_CSV)
df_oof["split_seed"] = df_oof["split_seed"].astype(int)
df_oof["fold"] = df_oof["fold"].astype(int)
df_oof["patient_id"] = df_oof["patient_id"].astype(str)
df_oof["y"] = df_oof["y"].astype(int)

# ogni paziente ha una fold per split_seed
pid_fold = (df_oof.groupby(["split_seed","patient_id"])["fold"].first().reset_index())
pid_y    = (df_oof.groupby("patient_id")["y"].first().reset_index())

# ---------- dataset ALL (labeled cohort) ----------
MANIFEST_PATIENT_LAB = "/hpcnfs/home/ieo7627/training_manifest_patient_COMPLETE_TITAN.csv"  # cambia se serve

# === 1) dataset base "completo" (una volta sola) ===
# Usa EXACT stessi argomenti con cui crei ds_train/ds_val nel training loop.
# Esempio (adatta i nomi variabile ai tuoi):
ds_base = PatientBagsDataset(
    manifest_csv=MANIFEST_PATIENT_LAB,
    norm_mode="slide_zscore",
    sample_per_patient=1024,
    # ... qualsiasi altro parametro che passi nel training (feature_root, tile_index, has_cont, ecc.)
)

# IMPORTANT: lo useremo sempre in modalità val deterministica (cambia solo fold=bag_fold)
ds_base.set_split("val", fold=0)

# === 2) mapping patient_id -> indice nel dataset ===
# (quasi sicuramente ds_base ha un df/manifest_df con patient_id)
if hasattr(ds_base, "df") and "patient_id" in ds_base.df.columns:
    all_pids = ds_base.df["patient_id"].astype(str).tolist()
elif hasattr(ds_base, "manifest_df") and "patient_id" in ds_base.manifest_df.columns:
    all_pids = ds_base.manifest_df["patient_id"].astype(str).tolist()
else:
    raise RuntimeError("Non trovo patient_id nel dataset base: aggiusta qui in base alla tua classe PatientBagsDataset.")

pid_to_idx = {pid:i for i,pid in enumerate(all_pids)}

ds_all=ds_base
# controlla che i pazienti OOF ci siano nel dataset
missing = [pid for pid in pid_y["patient_id"].tolist() if pid not in pid_to_idx]
print("[OOF-3x5] pazienti OOF:", len(pid_y), "| presenti nel dataset:", len(pid_y)-len(missing), "| missing:", len(missing))
if len(missing) > 0:
    print("Esempi missing:", missing[:10])

# ---------- RUN: per ogni split_seed, per ogni fold_model (checkpoint), predici SOLO i pazienti del suo fold val ----------
rows = []
for split_seed in SPLIT_SEEDS:
    for model_fold in MODEL_FOLDS:
        ckpt = f"{MIL_KIND}_split{split_seed}_init{INIT_SEED}_fold{model_fold}_best.pth"
        if not os.path.exists(ckpt):
            print("[WARN missing]", ckpt); continue

        # pazienti che per questo split_seed stanno nel fold == model_fold (OOF val set)
        val_pids = pid_fold[(pid_fold["split_seed"]==split_seed) & (pid_fold["fold"]==model_fold)]["patient_id"].tolist()
        val_idxs = [pid_to_idx[pid] for pid in val_pids if pid in pid_to_idx]
        if len(val_idxs) == 0:
            print("[WARN] no val idxs for", split_seed, model_fold); continue

        model = load_state_dict_flexible(build_model_for_inference(), ckpt)
        model.eval()

        for bag_fold in BAG_FOLDS:
            ds_all.set_split("val", fold=bag_fold)
            if hasattr(ds_all, "val_idx_cache"):
                ds_all.val_idx_cache = {}   # fondamentale: cambia bag deterministica

            loader = make_loader(ds_all, val_idxs)
            ys, ps, pids = infer_loader_test(model, loader, DEVICE)

            for y_i, p_i, pid_i in zip(ys, ps, pids):
                rows.append({
                    "patient_id": str(pid_i),
                    "y": int(y_i),
                    "split_seed": int(split_seed),
                    "model_fold": int(model_fold),
                    "bag_fold": int(bag_fold),
                    "p_raw": float(p_i),
                    "ckpt": ckpt,
                    "source": "OOF_fair"
                })

df_rows = pd.DataFrame(rows)
df_rows.to_csv(OUT_ROWS, index=False)
print("Saved:", OUT_ROWS, "shape:", df_rows.shape)

# ---------- Aggregazioni ----------
# per (patient, split_seed): aggrega 5 bag (logitmean)
per_seed = df_rows.groupby(["patient_id","split_seed"]).agg(
    y=("y","first"),
    p_seed=("p_raw", lambda x: logitmean(x.values)),
    p_std_bag=("p_raw","std"),
).reset_index()

# per paziente: aggrega 3 split_seed (median robusta)
per_patient = per_seed.groupby("patient_id").agg(
    y=("y","first"),
    p_final_mean=("p_seed","mean"),
    p_final_median=("p_seed","median"),
    p_final_logitmean=("p_seed", lambda x: logitmean(x.values)),
    std_due_to_model=("p_seed","std"),       # qui "model" = split_seed (3 modelli OOF)
    mean_std_due_to_bag=("p_std_bag","mean"),
).reset_index()

per_patient.to_csv(OUT_PAT, index=False)
print("Saved:", OUT_PAT, "shape:", per_patient.shape)

# ---------- Metriche OOF fair (con y vera) ----------
y = per_patient["y"].astype(int).values
print("\n[OOF FAIR METRICS]")
print("mean   :", metrics(y, per_patient["p_final_mean"].values))
print("median :", metrics(y, per_patient["p_final_median"].values))
print("logit  :", metrics(y, per_patient["p_final_logitmean"].values))

print("\n[OOF FAIR STABILITY]")
print("median mean_std_due_to_bag:", float(per_patient["mean_std_due_to_bag"].median()))
print("median std_due_to_model   :", float(per_patient["std_due_to_model"].median()))

# ---------- Confronto vs TEST (se fornito) ----------
summary_rows = []
summary_rows.append({
    "domain": "OOF_fair_3models",
    "metric_mean": str(metrics(y, per_patient["p_final_mean"].values)),
    "metric_median": str(metrics(y, per_patient["p_final_median"].values)),
    "metric_logit": str(metrics(y, per_patient["p_final_logitmean"].values)),
    "median_mean_std_due_to_bag": float(per_patient["mean_std_due_to_bag"].median()),
    "median_std_due_to_model": float(per_patient["std_due_to_model"].median()),
})

if TEST_ROWS_CSV is not None and os.path.exists(TEST_ROWS_CSV):
    df_test = pd.read_csv(TEST_ROWS_CSV)
    df_test["patient_id"] = df_test["patient_id"].astype(str)
    df_test["y"] = df_test["y"].astype(int)

    # ricostruisci dal test lo score migliore che avevamo visto: bag_mean -> median (15 modelli)
    # prima: per modello (split_seed,model_fold): mean sulle 5 bag
    test_pm = df_test.groupby(["patient_id","split_seed","model_fold"]).agg(
        y=("y","first"),
        p_model=("p_raw","mean"),
    ).reset_index()
    test_pat = test_pm.groupby("patient_id").agg(
        y=("y","first"),
        p_final=("p_model","median"),
        std_due_to_model=("p_model","std"),
    ).reset_index()

    yt = test_pat["y"].values.astype(int)
    mt = metrics(yt, test_pat["p_final"].values)
    summary_rows.append({
        "domain": "TEST_15models_bagMean->median",
        "metric_median": str(mt),
        "median_std_due_to_model": float(test_pat["std_due_to_model"].median()),
        "median_mean_std_due_to_bag": np.nan,  # lo hai già nel file test_per_patient se vuoi
    })

df_sum = pd.DataFrame(summary_rows)
df_sum.to_csv(OUT_SUM, index=False)
print("\nSaved:", OUT_SUM)
display(df_sum)


[OOF-3x5] pazienti OOF: 89 | presenti nel dataset: 89 | missing: 0
